Giai đoạn 1: Input Video từ YouTube
Input

Người dùng nhập:

YouTube URL hoặc Playlist URL
Output

Hệ thống trích xuất metadata:

{
  "video_id": "PTXkJbkGtUw",
  "youtube_url": "...",
  "title": "...",
  "duration": 3600,
  "channel": "...",
  "language": "vi/en/unknown",
  "playlist_id": null
}

Giai đoạn 2: Thu thập transcript
Mục tiêu

Lấy nội dung lời nói trong video làm nguồn tri thức chính.

Thứ tự ưu tiên
1. Subtitle chính thức từ YouTube
2. Auto subtitle từ YouTube
3. Whisper ASR fallback
Output chuẩn
[
  {
    "start": 1.76,
    "end": 3.55,
    "text": "Hôm nay chúng ta sẽ tìm hiểu về RAG..."
  }
]

In [13]:
!pip install -q yt-dlp youtube-transcript-api faster-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 4.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 50.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.3/36.3 MB 53.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 43.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 48.3 MB/s eta 0:00:0000:0100:01m


In [14]:
import os
import re
import json
from pathlib import Path
from urllib.parse import urlparse, parse_qs

import yt_dlp
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import (
    TranscriptsDisabled,
    NoTranscriptFound,
    VideoUnavailable
)

BASE_DIR = Path("/kaggle/working/video_rag_project")
DATA_DIR = BASE_DIR / "data"
AUDIO_DIR = BASE_DIR / "audio"

DATA_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("AUDIO_DIR:", AUDIO_DIR)

BASE_DIR: /kaggle/working/video_rag_project
DATA_DIR: /kaggle/working/video_rag_project/data
AUDIO_DIR: /kaggle/working/video_rag_project/audio


In [15]:
def extract_video_id(youtube_url: str) -> str:
    """
    Trích xuất video_id từ YouTube URL.
    Hỗ trợ:
    - youtube.com/watch?v=...
    - youtu.be/...
    - youtube.com/shorts/...
    """
    parsed = urlparse(youtube_url)

    if parsed.hostname in ["www.youtube.com", "youtube.com", "m.youtube.com"]:
        if parsed.path == "/watch":
            query = parse_qs(parsed.query)
            return query.get("v", [None])[0]

        if parsed.path.startswith("/shorts/"):
            return parsed.path.split("/shorts/")[-1].split("/")[0]

        if parsed.path.startswith("/embed/"):
            return parsed.path.split("/embed/")[-1].split("/")[0]

    if parsed.hostname in ["youtu.be", "www.youtu.be"]:
        return parsed.path.strip("/").split("/")[0]

    raise ValueError("Không lấy được video_id từ URL. Hãy kiểm tra lại link YouTube.")

Giai đoạn 1: Lấy metadata video

In [16]:
def extract_video_metadata(youtube_url: str) -> dict:
    """
    Lấy metadata cơ bản của video bằng yt-dlp.
    Không download video ở bước này.
    """
    ydl_opts = {
        "quiet": True,
        "skip_download": True,
        "extract_flat": False,
        "noplaylist": False,
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(youtube_url, download=False)

    # Trường hợp URL là playlist
    playlist_id = info.get("playlist_id")
    
    # Nếu link playlist, lấy video đầu tiên để demo phase 1
    if "entries" in info and info["entries"]:
        first_video = info["entries"][0]
        info = first_video

    video_id = info.get("id") or extract_video_id(youtube_url)

    metadata = {
        "video_id": video_id,
        "youtube_url": f"https://www.youtube.com/watch?v={video_id}",
        "title": info.get("title"),
        "duration": info.get("duration"),
        "channel": info.get("channel") or info.get("uploader"),
        "language": info.get("language") or "unknown",
        "playlist_id": playlist_id,
    }

    return metadata

In [17]:
# Test
youtube_url = input("Nhập YouTube URL: ").strip()

metadata = extract_video_metadata(youtube_url)

metadata_path = DATA_DIR / "video_metadata.json"
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

metadata

Nhập YouTube URL:  https://www.youtube.com/watch?v=ULE78ME1ckQ


{'video_id': 'ULE78ME1ckQ',
 'youtube_url': 'https://www.youtube.com/watch?v=ULE78ME1ckQ',
 'title': 'Tất cả các thuật toán Machine Learning trong 23 phút',
 'duration': 1558,
 'channel': 'Việt Nguyễn AI',
 'language': 'vi',
 'playlist_id': None}

In [18]:
# Cell 5 — Hàm chuẩn hóa transcript segment

# youtube-transcript-api thường trả về:

# {
#   "text": "...",
#   "start": 1.76,
#   "duration": 3.55
# }

# Nhưng pipeline của mình cần:

# {
#   "start": 1.76,
#   "end": 5.31,
#   "text": "..."
# }

# Nên cần hàm convert:

def normalize_transcript_segments(raw_segments: list) -> list:
    """
    Chuyển transcript từ dạng:
    {text, start, duration}
    
    sang dạng chuẩn:
    {start, end, text}
    """
    normalized = []

    for seg in raw_segments:
        start = float(seg.get("start", 0))
        duration = float(seg.get("duration", 0))
        end = start + duration
        text = seg.get("text", "").strip()

        if not text:
            continue

        normalized.append({
            "start": round(start, 2),
            "end": round(end, 2),
            "text": text
        })

    return normalized

Giai đoạn 2A: Lấy subtitle chính thức hoặc auto subtitle

In [19]:
def fetch_youtube_transcript(video_id: str, preferred_languages=None) -> dict:
    """
    Lấy transcript từ YouTube bằng youtube-transcript-api bản mới.

    Thứ tự ưu tiên:
    1. vi
    2. en

    Lưu ý:
    Bản mới không dùng:
        YouTubeTranscriptApi.list_transcripts(video_id)

    Mà dùng:
        ytt_api = YouTubeTranscriptApi()
        ytt_api.fetch(video_id, languages=[...])
    """
    if preferred_languages is None:
        preferred_languages = ["vi", "en"]

    try:
        ytt_api = YouTubeTranscriptApi()

        raw_segments = ytt_api.fetch(
            video_id,
            languages=preferred_languages
        )

        segments = []

        for seg in raw_segments:
            start = float(seg.start)
            duration = float(seg.duration)
            end = start + duration
            text = seg.text.strip()

            if not text:
                continue

            segments.append({
                "start": round(start, 2),
                "end": round(end, 2),
                "text": text
            })

        return {
            "segments": segments,
            "metadata": {
                
                "source": "youtube_subtitle_or_auto",
                "language_priority": ["vi", "en"],
                "detected_language": "unknown",
                "confidence": None
            }
                
                
            
        }

    except Exception as e:
        print("Không lấy được subtitle từ YouTube:", type(e).__name__)
        print("Chi tiết lỗi:", str(e))

        return {
            "segments": segments,
            "metadata": {
                "source": "youtube_subtitle_or_auto",
                "language_priority": preferred_languages,
                "detected_language": "unknown",
                "confidence": None
            }
        }

In [20]:
# Test

video_id = metadata["video_id"]

transcript_result = fetch_youtube_transcript(
    video_id=video_id,
    preferred_languages=["vi", "en"]
)

print("Transcript source:", transcript_result["metadata"])
print("Số segment:", len(transcript_result["segments"]))
print(transcript_result["segments"][:3])

Transcript source: {'source': 'youtube_subtitle_or_auto', 'language_priority': ['vi', 'en'], 'detected_language': 'unknown', 'confidence': None}
Số segment: 854
[{'start': 0.08, 'end': 3.24, 'text': 'Xin chào các bạn Nếu các bạn là những'}, {'start': 1.52, 'end': 4.96, 'text': 'người đang muốn bắt đầu tìm hiểu về ai'}, {'start': 3.24, 'end': 6.48, 'text': 'hay là machining nhưng các bạn chưa biết'}]


In [21]:
# Lưu transcript trước khi clean

video_id = metadata["video_id"]

raw_transcript_output = {
    "video_metadata": metadata,
    "transcript_metadata": transcript_result["metadata"],
    "segments": transcript_result["segments"]
}

raw_transcript_path = DATA_DIR / f"{video_id}_raw_transcript.json"

with open(raw_transcript_path, "w", encoding="utf-8") as f:
    json.dump(raw_transcript_output, f, ensure_ascii=False, indent=2)

print("Đã lưu raw transcript:", raw_transcript_path)

Đã lưu raw transcript: /kaggle/working/video_rag_project/data/ULE78ME1ckQ_raw_transcript.json


In [22]:
for i, seg in enumerate(transcript_result["segments"][:20]):
    print(f"{i+1:02d}. [{seg['start']} - {seg['end']}] {seg['text']}")

01. [0.08 - 3.24] Xin chào các bạn Nếu các bạn là những
02. [1.52 - 4.96] người đang muốn bắt đầu tìm hiểu về ai
03. [3.24 - 6.48] hay là machining nhưng các bạn chưa biết
04. [4.96 - 8.48] rằng mình nên bắt đầu từ đâu hoặc các
05. [6.48 - 10.0] bạn là những người đã tìm hiểu về ai về
06. [8.48 - 11.24] machining một thời gian rồi nhưng các
07. [10.0 - 12.72] bạn vẫn chưa thực sự hệ thống hóa được
08. [11.24 - 14.44] toàn bộ kiến thức của mình thì video này
09. [12.72 - 16.32] của mình là dành cho các bạn trong video
10. [14.44 - 18.08] này mình xin tổng hợp và tóm tắt toàn bộ
11. [16.32 - 19.68] các thật toán machin ling cơ bản và phổ
12. [18.08 - 22.0] biến nhất mà bất kỳ ai khi muốn học về
13. [19.68 - 23.6] ai học về machining đều cần phải nắm rõ
14. [22.0 - 25.12] do video này mình hướng đến đối tượng là
15. [23.6 - 26.48] cả những bạn chưa từng tiếp xúc với ai
16. [25.12 - 27.8] bao giờ do đó thì mình sẽ cố gắng giải
17. [26.48 - 29.24] thích tất cả mọi thứ ở mức đơn giản nhất
18

In [23]:
raw_text_preview = " ".join([seg["text"] for seg in transcript_result["segments"][:30]])
print(raw_text_preview)

Xin chào các bạn Nếu các bạn là những người đang muốn bắt đầu tìm hiểu về ai hay là machining nhưng các bạn chưa biết rằng mình nên bắt đầu từ đâu hoặc các bạn là những người đã tìm hiểu về ai về machining một thời gian rồi nhưng các bạn vẫn chưa thực sự hệ thống hóa được toàn bộ kiến thức của mình thì video này của mình là dành cho các bạn trong video này mình xin tổng hợp và tóm tắt toàn bộ các thật toán machin ling cơ bản và phổ biến nhất mà bất kỳ ai khi muốn học về ai học về machining đều cần phải nắm rõ do video này mình hướng đến đối tượng là cả những bạn chưa từng tiếp xúc với ai bao giờ do đó thì mình sẽ cố gắng giải thích tất cả mọi thứ ở mức đơn giản nhất có thể những cái thứ phức tạp như là công thức Toán học thì mình sẽ không đề cập đến trước khi đi vào nội dung chính chúng ta cần cùng nhau nhắc lại thế nào là machine learning hay machine learning là gì machine learning tiếng Việt là học máy đây là một lĩnh vực con của ai tập trung vào việc giúp cho máy tính có khả năng tự

Giai đoạn 2B: Whisper fallback nếu không có subtitle

In [24]:
def download_audio_for_whisper(youtube_url: str, video_id: str) -> Path:
    """
    Download audio từ YouTube để dùng Whisper ASR.
    Output file: /kaggle/working/video_rag_project/audio/{video_id}.mp3
    """
    output_template = str(AUDIO_DIR / f"{video_id}.%(ext)s")

    ydl_opts = {
        "format": "bestaudio/best",
        "outtmpl": output_template,
        "quiet": True,
        "noplaylist": True,
        "postprocessors": [
            {
                "key": "FFmpegExtractAudio",
                "preferredcodec": "mp3",
                "preferredquality": "192",
            }
        ],
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([youtube_url])

    audio_path = AUDIO_DIR / f"{video_id}.mp3"

    if not audio_path.exists():
        raise FileNotFoundError(f"Không tìm thấy audio file: {audio_path}")

    return audio_path

In [25]:
def transcribe_with_faster_whisper(audio_path: Path, language: str = None, model_size: str = "small") -> dict:
    """
    Dùng faster-whisper để ASR nếu video không có subtitle.
    
    model_size có thể dùng:
    - tiny
    - base
    - small
    - medium
    
    Trên Kaggle nên bắt đầu với small hoặc base.
    """
    from faster_whisper import WhisperModel

    # Nếu Kaggle có GPU T4 thì dùng cuda.
    # Nếu không có GPU, đổi device="cpu", compute_type="int8".
    try:
        model = WhisperModel(model_size, device="cuda", compute_type="float16")
    except Exception:
        model = WhisperModel(model_size, device="cpu", compute_type="int8")

    segments_iter, info = model.transcribe(
        str(audio_path),
        language=language,
        vad_filter=True,
        beam_size=5
    )

    segments = []
    for seg in segments_iter:
        text = seg.text.strip()
        if not text:
            continue

        segments.append({
            "start": round(float(seg.start), 2),
            "end": round(float(seg.end), 2),
            "text": text
        })

    return {
        "segments": segments,
        "metadata": {
            "source": "whisper",
            "language": info.language,
            "confidence": getattr(info, "language_probability", None)
        }
    }

In [26]:
def process_stage_1_2(youtube_url: str, preferred_languages=None, whisper_model_size: str = "small") -> dict:
    """
    Giai đoạn 1 + 2:
    - Lấy metadata video
    - Lấy transcript theo thứ tự:
        1. YouTube subtitle chính thức
        2. YouTube auto subtitle
        3. Whisper ASR fallback
    - Lưu output ra file JSON
    """
    if preferred_languages is None:
        preferred_languages = ["vi", "en"]

    # Stage 1: metadata
    metadata = extract_video_metadata(youtube_url)
    video_id = metadata["video_id"]

    # Stage 2A: YouTube subtitle / auto subtitle
    transcript_result = fetch_youtube_transcript(
        video_id=video_id,
        preferred_languages=preferred_languages
    )

    # Stage 2B: Whisper fallback
    if len(transcript_result["segments"]) == 0:
        print("Không có subtitle. Bắt đầu Whisper fallback...")

        audio_path = download_audio_for_whisper(
            youtube_url=metadata["youtube_url"],
            video_id=video_id
        )

        transcript_result = transcribe_with_faster_whisper(
            audio_path=audio_path,
            language=None,
            model_size=whisper_model_size
        )

    # Gộp output
    output = {
        "metadata": metadata,
        "transcript_metadata": transcript_result["metadata"],
        "transcript": transcript_result["segments"]
    }

    # Lưu file
    output_path = DATA_DIR / f"{video_id}_stage_1_2_output.json"
    metadata_path = DATA_DIR / f"{video_id}_metadata.json"
    transcript_path = DATA_DIR / f"{video_id}_raw_transcript.json"

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)

    with open(metadata_path, "w", encoding="utf-8") as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)

    with open(transcript_path, "w", encoding="utf-8") as f:
        json.dump({
            "transcript_metadata": transcript_result["metadata"],
            "segments": transcript_result["segments"]
        }, f, ensure_ascii=False, indent=2)

    print("Đã lưu:")
    print("-", output_path)
    print("-", metadata_path)
    print("-", transcript_path)

    return output

In [27]:
# Test 1+ 2


youtube_url = input("Nhập YouTube URL: ").strip()

stage_1_2_output = process_stage_1_2(
    youtube_url=youtube_url,
    preferred_languages=["vi", "en"],
    whisper_model_size="small"
)

print("Video metadata:")
print(json.dumps(stage_1_2_output["metadata"], ensure_ascii=False, indent=2))

print("\nTranscript metadata:")
print(json.dumps(stage_1_2_output["transcript_metadata"], ensure_ascii=False, indent=2))

print("\nSố transcript segments:", len(stage_1_2_output["transcript"]))

print("\n3 segment đầu:")
for seg in stage_1_2_output["transcript"][:10]:
    print(seg)

Nhập YouTube URL:  https://www.youtube.com/watch?v=ULE78ME1ckQ


Đã lưu:
- /kaggle/working/video_rag_project/data/ULE78ME1ckQ_stage_1_2_output.json
- /kaggle/working/video_rag_project/data/ULE78ME1ckQ_metadata.json
- /kaggle/working/video_rag_project/data/ULE78ME1ckQ_raw_transcript.json
Video metadata:
{
  "video_id": "ULE78ME1ckQ",
  "youtube_url": "https://www.youtube.com/watch?v=ULE78ME1ckQ",
  "title": "Tất cả các thuật toán Machine Learning trong 23 phút",
  "duration": 1558,
  "channel": "Việt Nguyễn AI",
  "language": "vi",
  "playlist_id": null
}

Transcript metadata:
{
  "source": "youtube_subtitle_or_auto",
  "language_priority": [
    "vi",
    "en"
  ],
  "detected_language": "unknown",
  "confidence": null
}

Số transcript segments: 854

3 segment đầu:
{'start': 0.08, 'end': 3.24, 'text': 'Xin chào các bạn Nếu các bạn là những'}
{'start': 1.52, 'end': 4.96, 'text': 'người đang muốn bắt đầu tìm hiểu về ai'}
{'start': 3.24, 'end': 6.48, 'text': 'hay là machining nhưng các bạn chưa biết'}
{'start': 4.96, 'end': 8.48, 'text': 'rằng mình nên

Giai đoạn 1: Input Video từ YouTube

Ở giai đoạn đầu tiên, hệ thống nhận đầu vào là một đường dẫn YouTube do người dùng cung cấp. Từ đường dẫn này, hệ thống sử dụng thư viện yt-dlp để trích xuất các thông tin metadata cơ bản của video như mã video, tiêu đề, thời lượng, tên kênh, ngôn ngữ và mã playlist nếu có. Các thông tin này được lưu lại nhằm phục vụ cho các giai đoạn xử lý tiếp theo như tóm tắt, chia đoạn, truy xuất và hiển thị kết quả.

Giai đoạn 2: Thu thập transcript

Sau khi có metadata, hệ thống tiến hành thu thập transcript của video. Transcript được xem là nguồn tri thức chính trong giai đoạn đầu của hệ thống RAG. Quy trình thu thập transcript được thực hiện theo thứ tự ưu tiên: đầu tiên là subtitle chính thức từ YouTube, tiếp theo là auto subtitle do YouTube tạo tự động, và cuối cùng là sử dụng mô hình Whisper để nhận dạng giọng nói nếu video không có subtitle. Kết quả transcript được chuẩn hóa về dạng danh sách các đoạn có start, end và text. Ngoài nội dung transcript, hệ thống cũng lưu thêm metadata gồm nguồn transcript, ngôn ngữ và độ tin cậy nếu có. Thông tin này giúp đánh giá chất lượng dữ liệu đầu vào ở các bước sau.

Giai đoạn 3: Làm sạch transcript
Mục tiêu

Loại bỏ dữ liệu nhiễu trước khi chunking và embedding.

Các bước xử lý
- Xóa HTML tag.
- Xóa tag rác như [Music], [Applause].
- Xóa dòng rỗng.
- Xóa subtitle bị lặp.
- Xóa đoạn tăng dần do auto subtitle.
- Chuẩn hóa khoảng trắng.
- Chuẩn hóa timestamp.
- Sửa start/end bị lệch.
Output
[
  {
    "start": 1.76,
    "end": 16.23,
    "text": "Hôm nay chúng ta sẽ tìm hiểu về Retrieval Augmented Generation..."
  }
]

In [28]:
# Hàm làm sạch text trong từng segment

import re
import html
import json
from pathlib import Path

def clean_segment_text(text: str) -> str:
    """
    Làm sạch nội dung text của 1 transcript segment.
    """
    if text is None:
        return ""

    # Decode HTML entity: &amp;, &#39;, ...
    text = html.unescape(text)

    # Xóa HTML tag nếu có
    text = re.sub(r"<[^>]+>", " ", text)

    # Xóa các tag rác thường gặp trong subtitle
    noise_patterns = [
        r"\[music\]",
        r"\[applause\]",
        r"\[laughter\]",
        r"\[sound\]",
        r"\[noise\]",
        r"\(music\)",
        r"\(applause\)",
        r"\(laughter\)",
    ]

    for pattern in noise_patterns:
        text = re.sub(pattern, " ", text, flags=re.IGNORECASE)

    # Xóa ký tự xuống dòng trong segment
    text = text.replace("\n", " ")

    # Chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [29]:
# Hàm kiểm tra duplicate subtitle

def normalize_for_duplicate_check(text: str) -> str:
    """
    Chuẩn hóa text để kiểm tra trùng lặp.
    """
    text = text.lower().strip()
    text = re.sub(r"[^\w\sÀ-ỹ]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text

def is_duplicate_text(current_text: str, previous_text: str) -> bool:
    """
    Kiểm tra 2 đoạn text có bị trùng không.
    Ở bước này chỉ xử lý trùng đơn giản để tránh xóa nhầm.
    """
    cur = normalize_for_duplicate_check(current_text)
    prev = normalize_for_duplicate_check(previous_text)

    if not cur or not prev:
        return False

    # Trùng hoàn toàn
    if cur == prev:
        return True

    # Một câu rất ngắn bị lặp lại trong câu sau
    if len(cur.split()) <= 4 and cur in prev:
        return True

    if len(prev.split()) <= 4 and prev in cur:
        return True

    return False

In [30]:
# Hàm sửa timestamp bị lỗi nhẹ
# Segment sau bắt đầu trước khi segment trước kết thúc. Nhưng đây là chuyện bình thường của YouTube auto subtitle. Không nên sửa quá mạnh, vì có thể mất timestamp gốc.

# Cách xử lý ở đây:

# - Nếu end <= start thì sửa end = start + 0.5
# - Nếu start bị âm thì đưa về 0
# - Nếu overlap nhẹ thì vẫn giữ nguyên
# - Không ép segment sau phải bắt đầu sau segment trước

def fix_segment_time(start, end, min_duration: float = 0.3):
    """
    Sửa timestamp lỗi cơ bản.
    Không xử lý overlap mạnh ở đây để giữ timestamp gốc.
    """
    try:
        start = float(start)
        end = float(end)
    except Exception:
        start = 0.0
        end = min_duration

    if start < 0:
        start = 0.0

    if end <= start:
        end = start + min_duration

    return round(start, 2), round(end, 2)

In [31]:
# Hàm clean chính

def clean_transcript_segments(raw_segments: list) -> list:
    """
    Làm sạch raw transcript segments.

    Input:
    [
        {"start": ..., "end": ..., "text": "..."}
    ]

    Output:
    [
        {"start": ..., "end": ..., "text": "..."}
    ]
    """
    clean_segments = []
    previous_text = ""

    for seg in raw_segments:
        start = seg.get("start", 0)
        end = seg.get("end", 0)
        text = seg.get("text", "")

        start, end = fix_segment_time(start, end)
        text = clean_segment_text(text)

        # Bỏ segment rỗng
        if not text:
            continue

        # Bỏ duplicate liền kề
        if previous_text and is_duplicate_text(text, previous_text):
            continue

        clean_segments.append({
            "start": start,
            "end": end,
            "text": text
        })

        previous_text = text

    return clean_segments

In [32]:
# Chạy Giai đoạn 3 trên transcript vừa lấy


clean_segments = clean_transcript_segments(transcript_result["segments"])

print("Số segment trước cleaning:", len(transcript_result["segments"]))
print("Số segment sau cleaning:", len(clean_segments))

print("\n10 segment sau cleaning:")
for i, seg in enumerate(clean_segments[:10]):
    print(f"{i+1:02d}. [{seg['start']} - {seg['end']}] {seg['text']}")

Số segment trước cleaning: 854
Số segment sau cleaning: 854

10 segment sau cleaning:
01. [0.08 - 3.24] Xin chào các bạn Nếu các bạn là những
02. [1.52 - 4.96] người đang muốn bắt đầu tìm hiểu về ai
03. [3.24 - 6.48] hay là machining nhưng các bạn chưa biết
04. [4.96 - 8.48] rằng mình nên bắt đầu từ đâu hoặc các
05. [6.48 - 10.0] bạn là những người đã tìm hiểu về ai về
06. [8.48 - 11.24] machining một thời gian rồi nhưng các
07. [10.0 - 12.72] bạn vẫn chưa thực sự hệ thống hóa được
08. [11.24 - 14.44] toàn bộ kiến thức của mình thì video này
09. [12.72 - 16.32] của mình là dành cho các bạn trong video
10. [14.44 - 18.08] này mình xin tổng hợp và tóm tắt toàn bộ


In [33]:
# Lưu clean transcript ra file JSON
video_id = metadata["video_id"]

clean_transcript_output = {
    "video_metadata": metadata,
    "transcript_metadata": {
        **transcript_result["metadata"],
        "cleaning": {
            "remove_noise_tags": True,
            "remove_html_tags": True,
            "normalize_whitespace": True,
            "remove_empty_segments": True,
            "remove_adjacent_duplicates": True,
            "fix_invalid_timestamp": True
        },
        "num_raw_segments": len(transcript_result["segments"]),
        "num_clean_segments": len(clean_segments)
    },
    "segments": clean_segments
}

clean_transcript_path = DATA_DIR / f"{video_id}_clean_transcript.json"

with open(clean_transcript_path, "w", encoding="utf-8") as f:
    json.dump(clean_transcript_output, f, ensure_ascii=False, indent=2)

print("Đã lưu clean transcript:", clean_transcript_path)

Đã lưu clean transcript: /kaggle/working/video_rag_project/data/ULE78ME1ckQ_clean_transcript.json


In [34]:
# Kiểm tra nhanh chất lượng sau cleaning
def preview_segments(segments, n=20):
    for i, seg in enumerate(segments[:n]):
        print(f"{i+1:02d}. [{seg['start']} - {seg['end']}] {seg['text']}")
preview_segments(clean_segments, n=20)


# Xem văn bản nối lại

clean_text_preview = " ".join([seg["text"] for seg in clean_segments[:40]])
print(clean_text_preview)

01. [0.08 - 3.24] Xin chào các bạn Nếu các bạn là những
02. [1.52 - 4.96] người đang muốn bắt đầu tìm hiểu về ai
03. [3.24 - 6.48] hay là machining nhưng các bạn chưa biết
04. [4.96 - 8.48] rằng mình nên bắt đầu từ đâu hoặc các
05. [6.48 - 10.0] bạn là những người đã tìm hiểu về ai về
06. [8.48 - 11.24] machining một thời gian rồi nhưng các
07. [10.0 - 12.72] bạn vẫn chưa thực sự hệ thống hóa được
08. [11.24 - 14.44] toàn bộ kiến thức của mình thì video này
09. [12.72 - 16.32] của mình là dành cho các bạn trong video
10. [14.44 - 18.08] này mình xin tổng hợp và tóm tắt toàn bộ
11. [16.32 - 19.68] các thật toán machin ling cơ bản và phổ
12. [18.08 - 22.0] biến nhất mà bất kỳ ai khi muốn học về
13. [19.68 - 23.6] ai học về machining đều cần phải nắm rõ
14. [22.0 - 25.12] do video này mình hướng đến đối tượng là
15. [23.6 - 26.48] cả những bạn chưa từng tiếp xúc với ai
16. [25.12 - 27.8] bao giờ do đó thì mình sẽ cố gắng giải
17. [26.48 - 29.24] thích tất cả mọi thứ ở mức đơn giản nhất
18

In [35]:
# Thống kê sau clean

def transcript_statistics(segments: list) -> dict:
    """
    Tính thống kê cơ bản cho transcript.
    """
    if not segments:
        return {
            "num_segments": 0,
            "total_words": 0,
            "total_duration": 0,
            "avg_words_per_segment": 0,
            "avg_duration_per_segment": 0
        }

    total_words = sum(len(seg["text"].split()) for seg in segments)
    total_duration = max(seg["end"] for seg in segments) - min(seg["start"] for seg in segments)

    avg_words = total_words / len(segments)
    avg_duration = sum(seg["end"] - seg["start"] for seg in segments) / len(segments)

    return {
        "num_segments": len(segments),
        "total_words": total_words,
        "total_duration": round(total_duration, 2),
        "avg_words_per_segment": round(avg_words, 2),
        "avg_duration_per_segment": round(avg_duration, 2)
    }

In [36]:
raw_stats = transcript_statistics(transcript_result["segments"])
clean_stats = transcript_statistics(clean_segments)

print("Raw transcript stats:")
print(json.dumps(raw_stats, ensure_ascii=False, indent=2))

print("\nClean transcript stats:")
print(json.dumps(clean_stats, ensure_ascii=False, indent=2))

Raw transcript stats:
{
  "num_segments": 854,
  "total_words": 7760,
  "total_duration": 1560.16,
  "avg_words_per_segment": 9.09,
  "avg_duration_per_segment": 3.62
}

Clean transcript stats:
{
  "num_segments": 854,
  "total_words": 7760,
  "total_duration": 1560.16,
  "avg_words_per_segment": 9.09,
  "avg_duration_per_segment": 3.62
}


Giai đoạn Transcript Cleaning được thực hiện nhằm chuẩn hóa dữ liệu transcript thô thu được từ YouTube hoặc Whisper. Do subtitle tự động thường bị nhiễu, có thể chứa ký tự HTML, tag âm thanh như [Music], dòng rỗng, khoảng trắng dư thừa hoặc các đoạn bị lặp, hệ thống tiến hành làm sạch từng segment trước khi đưa vào các bước xử lý tiếp theo.

Mỗi segment được chuẩn hóa về định dạng gồm start, end và text. Hệ thống loại bỏ các segment rỗng, chuẩn hóa khoảng trắng, xóa các tag không mang thông tin nội dung và kiểm tra các lỗi timestamp cơ bản như end nhỏ hơn hoặc bằng start. Kết quả sau cleaning được lưu thành clean_transcript.json để phục vụ cho giai đoạn tái cấu trúc transcript và chia chunk cho RAG.

Giai đoạn 4: Transcript Reconstruction.

Mục tiêu:

Input:
clean_segments

Output:
semantic_units

Tác dụng:
- Ghép các subtitle vụn thành đoạn có nghĩa hơn
- Giữ timestamp bắt đầu và kết thúc
- Mỗi đoạn dài khoảng 10–25 giây hoặc 60–120 từ
- Làm dữ liệu tốt hơn cho chunking, summarization, embedding và RAG

Với kết quả của bạn, mỗi segment trung bình chỉ khoảng 8.77 từ và 3.63 giây, quá ngắn để embedding tốt. Vì vậy cần ghép lại.

In [37]:
def count_words(text: str) -> int:
    """
    Đếm số từ trong một đoạn text.
    """
    if not text:
        return 0
    return len(text.split())

In [38]:
# Ghép transcript thành 1 đoạn text

def reconstruct_transcript_units(
    segments: list,
    min_duration: float = 10.0,
    max_duration: float = 25.0,
    min_words: int = 50,
    max_words: int = 120
) -> list:
    """
    Ghép subtitle segment ngắn thành semantic units.

    Input:
    [
        {"start": 2.84, "end": 6.64, "text": "..."}
    ]

    Output:
    [
        {
            "unit_id": "unit_0001",
            "start": 2.84,
            "end": 25.72,
            "text": "...",
            "num_words": 95,
            "duration": 22.88
        }
    ]
    """

    units = []

    current_texts = []
    current_start = None
    current_end = None

    for seg in segments:
        text = seg.get("text", "").strip()
        if not text:
            continue

        start = float(seg.get("start", 0))
        end = float(seg.get("end", start))

        if current_start is None:
            current_start = start

        current_end = end
        current_texts.append(text)

        merged_text = " ".join(current_texts)
        duration = current_end - current_start
        num_words = count_words(merged_text)

        # Kiểm tra xem đoạn hiện tại có nên được đóng lại không
        enough_duration = duration >= min_duration
        enough_words = num_words >= min_words
        too_long_duration = duration >= max_duration
        too_many_words = num_words >= max_words

        # Dấu hiệu kết thúc câu, nếu transcript có dấu câu
        sentence_end = merged_text.strip().endswith((".", "?", "!", "ạ", "nhé", "rồi"))

        should_close = False

        # Đóng nếu đã đủ dài và có dấu hiệu kết thúc tự nhiên
        if enough_duration and enough_words and sentence_end:
            should_close = True

        # Đóng bắt buộc nếu quá dài
        if too_long_duration or too_many_words:
            should_close = True

        if should_close:
            unit_id = f"unit_{len(units) + 1:04d}"

            units.append({
                "unit_id": unit_id,
                "start": round(current_start, 2),
                "end": round(current_end, 2),
                "text": merged_text.strip(),
                "num_words": num_words,
                "duration": round(duration, 2)
            })

            current_texts = []
            current_start = None
            current_end = None

    # Thêm phần còn lại nếu có
    if current_texts:
        merged_text = " ".join(current_texts)
        duration = current_end - current_start
        num_words = count_words(merged_text)

        unit_id = f"unit_{len(units) + 1:04d}"

        units.append({
            "unit_id": unit_id,
            "start": round(current_start, 2),
            "end": round(current_end, 2),
            "text": merged_text.strip(),
            "num_words": num_words,
            "duration": round(duration, 2)
        })

    return units

In [39]:
# Recontruction

semantic_units = reconstruct_transcript_units(
    clean_segments,
    min_duration=10.0,
    max_duration=25.0,
    min_words=50,
    max_words=120
)

print("Số clean segments:", len(clean_segments))
print("Số semantic units:", len(semantic_units))

print("\n5 semantic units đầu:")
for unit in semantic_units[:5]:
    print("=" * 80)
    print(f"{unit['unit_id']} | {unit['start']} - {unit['end']} | {unit['num_words']} words | {unit['duration']}s")
    print(unit["text"])

Số clean segments: 854
Số semantic units: 68

5 semantic units đầu:
unit_0001 | 0.08 - 25.12 | 126 words | 25.04s
Xin chào các bạn Nếu các bạn là những người đang muốn bắt đầu tìm hiểu về ai hay là machining nhưng các bạn chưa biết rằng mình nên bắt đầu từ đâu hoặc các bạn là những người đã tìm hiểu về ai về machining một thời gian rồi nhưng các bạn vẫn chưa thực sự hệ thống hóa được toàn bộ kiến thức của mình thì video này của mình là dành cho các bạn trong video này mình xin tổng hợp và tóm tắt toàn bộ các thật toán machin ling cơ bản và phổ biến nhất mà bất kỳ ai khi muốn học về ai học về machining đều cần phải nắm rõ do video này mình hướng đến đối tượng là
unit_0002 | 23.6 - 49.52 | 123 words | 25.92s
cả những bạn chưa từng tiếp xúc với ai bao giờ do đó thì mình sẽ cố gắng giải thích tất cả mọi thứ ở mức đơn giản nhất có thể những cái thứ phức tạp như là công thức Toán học thì mình sẽ không đề cập đến trước khi đi vào nội dung chính chúng ta cần cùng nhau nhắc lại thế nào là machi

In [40]:
# Thống kê sematic

def semantic_unit_statistics(units: list) -> dict:
    """
    Thống kê các semantic units sau reconstruction.
    """
    if not units:
        return {
            "num_units": 0,
            "total_words": 0,
            "avg_words_per_unit": 0,
            "avg_duration_per_unit": 0,
            "min_words": 0,
            "max_words": 0,
            "min_duration": 0,
            "max_duration": 0
        }

    word_counts = [unit["num_words"] for unit in units]
    durations = [unit["duration"] for unit in units]

    return {
        "num_units": len(units),
        "total_words": sum(word_counts),
        "avg_words_per_unit": round(sum(word_counts) / len(word_counts), 2),
        "avg_duration_per_unit": round(sum(durations) / len(durations), 2),
        "min_words": min(word_counts),
        "max_words": max(word_counts),
        "min_duration": round(min(durations), 2),
        "max_duration": round(max(durations), 2)
    }

In [41]:
unit_stats = semantic_unit_statistics(semantic_units)

print(json.dumps(unit_stats, ensure_ascii=False, indent=2))

{
  "num_units": 68,
  "total_words": 7760,
  "avg_words_per_unit": 114.12,
  "avg_duration_per_unit": 24.78,
  "min_words": 22,
  "max_words": 129,
  "min_duration": 5.4,
  "max_duration": 28.24
}


In [42]:
# Lưu semantic units ra json

video_id = metadata["video_id"]

semantic_units_output = {
    "video_metadata": metadata,
    "transcript_metadata": {
        **transcript_result["metadata"],
        "reconstruction": {
            "method": "timestamp_word_based_grouping",
            "min_duration": 10.0,
            "max_duration": 25.0,
            "min_words": 50,
            "max_words": 120
        },
        "num_clean_segments": len(clean_segments),
        "num_semantic_units": len(semantic_units)
    },
    "semantic_units": semantic_units
}

semantic_units_path = DATA_DIR / f"{video_id}_semantic_units.json"

with open(semantic_units_path, "w", encoding="utf-8") as f:
    json.dump(semantic_units_output, f, ensure_ascii=False, indent=2)

print("Đã lưu semantic units:", semantic_units_path)

Đã lưu semantic units: /kaggle/working/video_rag_project/data/ULE78ME1ckQ_semantic_units.json


In [43]:
def preview_semantic_units(units: list, start_index: int = 0, n: int = 5):
    """
    In semantic units để kiểm tra.
    """
    selected_units = units[start_index:start_index + n]

    for unit in selected_units:
        print("=" * 100)
        print(f"{unit['unit_id']} | {unit['start']}s - {unit['end']}s | {unit['num_words']} words | {unit['duration']}s")
        print(unit["text"])

In [44]:
preview_semantic_units(semantic_units, start_index=10, n=5)

unit_0011 | 225.28s - 250.32s | 127 words | 25.04s
nhằm dự đoán mức lương của một nhân viên dựa vào Số năm kinh nghiệm mà anh ta có 10 chấm màu xanh lá cây ở đây tương ứng với 10 nhân viên mà chúng ta đã biết họ có bao nhiêu năm kinh nghiệm làm việc cũng như là mức lương hiện tại của họ và chúng ta sẽ dùng những thông tin này để xây dựng mô hình dự đoán thế con đườ thẳng màu đỏ ở đây chính là kết quả của thuật toán linear rection khi chúng ta huấn luyện mô hình dựa trên 10 điểm dữ liệu mà chúng ta có đường thẳng này là đường thẳng tốt nhất đi qua dữ liệu của chúng ta vì khoảng cách trung bình từ
unit_0012 | 248.8s - 275.12s | 129 words | 26.32s
các điểm dữ liệu đến đường thẳng này là tối thiểu trong ví dụ của chúng ta thì Số năm kinh nghiệm là x còn mức lương là y giả sử chúng ta đã tìm được một quan hệ tuyến tính giữa Số năm kinh nghiệm và mức lương là mức lương thì bằng 2 triệu nhân với Số năm kinh nghiệm cộng với 8 triệu Thế thì sau này khi mà có những nhân viên mới vào công ty mà c

In [45]:
semantic_units_txt_path = DATA_DIR / f"{video_id}_semantic_units_preview.txt"

with open(semantic_units_txt_path, "w", encoding="utf-8") as f:
    for unit in semantic_units:
        f.write("=" * 100 + "\n")
        f.write(f"{unit['unit_id']} | {unit['start']}s - {unit['end']}s | {unit['num_words']} words | {unit['duration']}s\n")
        f.write(unit["text"] + "\n\n")

print("Đã lưu file preview:", semantic_units_txt_path)

Đã lưu file preview: /kaggle/working/video_rag_project/data/ULE78ME1ckQ_semantic_units_preview.txt


Sau bước làm sạch transcript, các subtitle vẫn còn ở dạng các đoạn ngắn do YouTube tự động tạo ra. Điều này gây khó khăn cho quá trình tóm tắt và truy xuất ngữ nghĩa vì mỗi segment chỉ chứa một phần nhỏ của câu nói. Vì vậy, hệ thống thực hiện bước Transcript Reconstruction nhằm ghép các segment liên tiếp thành các semantic units có độ dài phù hợp.

Việc ghép đoạn được thực hiện dựa trên các tiêu chí về thời lượng, số lượng từ và thứ tự thời gian. Mỗi semantic unit lưu lại timestamp bắt đầu, timestamp kết thúc, nội dung văn bản, số lượng từ và thời lượng. Kết quả của bước này giúp nội dung transcript có ngữ cảnh đầy đủ hơn, hỗ trợ tốt hơn cho các bước chunking, embedding, summarization và RAG QA.


Tức là có lúc semantic unit bị cắt giữa cụm từ. Điều này không quá nghiêm trọng, vì ở bước chunking mình sẽ dùng overlap theo unit. Ví dụ chunk sau sẽ lặp lại 1 unit trước đó để tránh mất ngữ cảnh.

Giai đoạn 5: Hybrid Chunking cho RAG
Mục tiêu

Từ semantic_units, tạo ra các chunk lớn hơn, phù hợp cho embedding và RAG.

Cấu hình đề xuất:

chunk_size: 300–500 từ
overlap: 1 semantic unit
max_duration: 2–4 phút / chunk

Với dữ liệu của bạn:

1 semantic unit ≈ 110 từ
1 semantic unit ≈ 25 giây

Vậy mỗi chunk nên gồm khoảng:

3–4 semantic units
≈ 330–440 từ
≈ 75–110 giây

Đây là kích thước rất phù hợp cho RAG.

In [46]:
# Tạo hybrid chunk

def create_hybrid_chunks(
    semantic_units: list,
    video_id: str,
    target_words: int = 400,
    min_words: int = 280,
    max_words: int = 500,
    max_duration: float = 240.0,
    overlap_units: int = 1
) -> list:
    """
    Tạo hybrid chunks từ semantic units.

    Ý tưởng:
    - Mỗi chunk gồm nhiều semantic units liên tiếp.
    - Chunk có khoảng 300–500 từ.
    - Có overlap 1 semantic unit giữa các chunk để tránh mất ngữ cảnh.
    - Giữ timestamp start/end để trả lời câu hỏi có mốc thời gian.

    Input:
    semantic_units = [
        {
            "unit_id": "unit_0001",
            "start": 2.84,
            "end": 28.0,
            "text": "...",
            "num_words": 118,
            "duration": 25.16
        }
    ]

    Output:
    chunks = [
        {
            "chunk_id": "...",
            "video_id": "...",
            "start": 2.84,
            "end": 99.28,
            "text": "...",
            "source": "transcript",
            "chapter": "auto",
            "num_words": 463,
            "duration": 96.44,
            "unit_ids": [...]
        }
    ]
    """

    chunks = []
    i = 0
    n = len(semantic_units)

    while i < n:
        current_units = []
        current_words = 0
        current_start = None
        current_end = None

        j = i

        while j < n:
            unit = semantic_units[j]

            unit_text = unit.get("text", "").strip()
            unit_words = int(unit.get("num_words", len(unit_text.split())))
            unit_start = float(unit.get("start", 0))
            unit_end = float(unit.get("end", unit_start))

            if current_start is None:
                current_start = unit_start

            next_words = current_words + unit_words
            next_duration = unit_end - current_start

            # Nếu thêm unit này làm chunk quá dài thì dừng,
            # nhưng chỉ dừng nếu chunk hiện tại đã đạt min_words.
            if current_units and next_words > max_words and current_words >= min_words:
                break

            if current_units and next_duration > max_duration and current_words >= min_words:
                break

            current_units.append(unit)
            current_words = next_words
            current_end = unit_end

            # Nếu đã đạt target_words thì đóng chunk
            if current_words >= target_words:
                break

            j += 1

        if not current_units:
            break

        chunk_text = " ".join([u["text"].strip() for u in current_units if u.get("text", "").strip()])
        chunk_words = len(chunk_text.split())
        chunk_start = float(current_units[0]["start"])
        chunk_end = float(current_units[-1]["end"])
        chunk_duration = chunk_end - chunk_start

        chunk_id = f"{video_id}_chunk_{len(chunks) + 1:04d}"

        chunks.append({
            "chunk_id": chunk_id,
            "video_id": video_id,
            "start": round(chunk_start, 2),
            "end": round(chunk_end, 2),
            "text": chunk_text,
            "source": "transcript",
            "chapter": "auto",
            "num_words": chunk_words,
            "duration": round(chunk_duration, 2),
            "unit_ids": [u["unit_id"] for u in current_units],
            "unit_start": current_units[0]["unit_id"],
            "unit_end": current_units[-1]["unit_id"]
        })

        # Dịch con trỏ sang chunk tiếp theo, có overlap
        if overlap_units > 0:
            next_i = max(j + 1 - overlap_units, i + 1)
        else:
            next_i = j + 1

        i = next_i

    return chunks

In [47]:
# Run

video_id = metadata["video_id"]

rag_chunks = create_hybrid_chunks(
    semantic_units=semantic_units,
    video_id=video_id,
    target_words=400,
    min_words=280,
    max_words=500,
    max_duration=240.0,
    overlap_units=1
)

print("Số semantic units:", len(semantic_units))
print("Số RAG chunks:", len(rag_chunks))

print("\n3 chunks đầu:")
for chunk in rag_chunks[:3]:
    print("=" * 100)
    print(f"{chunk['chunk_id']} | {chunk['start']}s - {chunk['end']}s")
    print(f"{chunk['num_words']} words | {chunk['duration']}s")
    print("Units:", chunk["unit_ids"])
    print(chunk["text"][:1000], "...")

Số semantic units: 68
Số RAG chunks: 22

3 chunks đầu:
ULE78ME1ckQ_chunk_0001 | 0.08s - 92.76s
489 words | 92.68s
Units: ['unit_0001', 'unit_0002', 'unit_0003', 'unit_0004']
Xin chào các bạn Nếu các bạn là những người đang muốn bắt đầu tìm hiểu về ai hay là machining nhưng các bạn chưa biết rằng mình nên bắt đầu từ đâu hoặc các bạn là những người đã tìm hiểu về ai về machining một thời gian rồi nhưng các bạn vẫn chưa thực sự hệ thống hóa được toàn bộ kiến thức của mình thì video này của mình là dành cho các bạn trong video này mình xin tổng hợp và tóm tắt toàn bộ các thật toán machin ling cơ bản và phổ biến nhất mà bất kỳ ai khi muốn học về ai học về machining đều cần phải nắm rõ do video này mình hướng đến đối tượng là cả những bạn chưa từng tiếp xúc với ai bao giờ do đó thì mình sẽ cố gắng giải thích tất cả mọi thứ ở mức đơn giản nhất có thể những cái thứ phức tạp như là công thức Toán học thì mình sẽ không đề cập đến trước khi đi vào nội dung chính chúng ta cần cùng nhau nhắc lại th

In [48]:
# Thống kê chunk

def chunk_statistics(chunks: list) -> dict:
    """
    Thống kê cơ bản các chunks.
    """
    if not chunks:
        return {
            "num_chunks": 0,
            "total_words": 0,
            "avg_words_per_chunk": 0,
            "avg_duration_per_chunk": 0,
            "min_words": 0,
            "max_words": 0,
            "min_duration": 0,
            "max_duration": 0
        }

    word_counts = [chunk["num_words"] for chunk in chunks]
    durations = [chunk["duration"] for chunk in chunks]

    return {
        "num_chunks": len(chunks),
        "total_words_with_overlap": sum(word_counts),
        "avg_words_per_chunk": round(sum(word_counts) / len(word_counts), 2),
        "avg_duration_per_chunk": round(sum(durations) / len(durations), 2),
        "min_words": min(word_counts),
        "max_words": max(word_counts),
        "min_duration": round(min(durations), 2),
        "max_duration": round(max(durations), 2)
    }

In [49]:
rag_chunk_stats = chunk_statistics(rag_chunks)

print(json.dumps(rag_chunk_stats, ensure_ascii=False, indent=2))

{
  "num_chunks": 22,
  "total_words_with_overlap": 10202,
  "avg_words_per_chunk": 463.73,
  "avg_duration_per_chunk": 95.17,
  "min_words": 355,
  "max_words": 499,
  "min_duration": 77.28,
  "max_duration": 111.68
}


In [50]:
# Save ra json

video_id = metadata["video_id"]

rag_chunks_output = {
    "video_metadata": metadata,
    "transcript_metadata": {
        **transcript_result["metadata"],
        "chunking": {
            "method": "hybrid_timestamp_semantic_word_overlap",
            "target_words": 400,
            "min_words": 280,
            "max_words": 500,
            "max_duration": 240.0,
            "overlap_units": 1
        },
        "num_semantic_units": len(semantic_units),
        "num_chunks": len(rag_chunks)
    },
    "chunks": rag_chunks
}

rag_chunks_path = DATA_DIR / f"{video_id}_rag_chunks.json"

with open(rag_chunks_path, "w", encoding="utf-8") as f:
    json.dump(rag_chunks_output, f, ensure_ascii=False, indent=2)

print("Đã lưu RAG chunks:", rag_chunks_path)

Đã lưu RAG chunks: /kaggle/working/video_rag_project/data/ULE78ME1ckQ_rag_chunks.json


In [51]:
rag_chunks_txt_path = DATA_DIR / f"{video_id}_rag_chunks_preview.txt"

with open(rag_chunks_txt_path, "w", encoding="utf-8") as f:
    for chunk in rag_chunks:
        f.write("=" * 120 + "\n")
        f.write(f"{chunk['chunk_id']} | {chunk['start']}s - {chunk['end']}s | {chunk['num_words']} words | {chunk['duration']}s\n")
        f.write(f"Units: {', '.join(chunk['unit_ids'])}\n")
        f.write("-" * 120 + "\n")
        f.write(chunk["text"] + "\n\n")

print("Đã lưu file preview:", rag_chunks_txt_path)

Đã lưu file preview: /kaggle/working/video_rag_project/data/ULE78ME1ckQ_rag_chunks_preview.txt


In [52]:
def preview_chunks(chunks: list, start_index: int = 0, n: int = 3, max_chars: int = 1200):
    """
    In chunks để kiểm tra.
    """
    selected_chunks = chunks[start_index:start_index + n]

    for chunk in selected_chunks:
        print("=" * 120)
        print(f"{chunk['chunk_id']} | {chunk['start']}s - {chunk['end']}s")
        print(f"{chunk['num_words']} words | {chunk['duration']}s")
        print(f"Units: {chunk['unit_ids']}")
        print("-" * 120)
        print(chunk["text"][:max_chars])
        if len(chunk["text"]) > max_chars:
            print("...")

preview_chunks(rag_chunks, start_index=0, n=3)

ULE78ME1ckQ_chunk_0001 | 0.08s - 92.76s
489 words | 92.68s
Units: ['unit_0001', 'unit_0002', 'unit_0003', 'unit_0004']
------------------------------------------------------------------------------------------------------------------------
Xin chào các bạn Nếu các bạn là những người đang muốn bắt đầu tìm hiểu về ai hay là machining nhưng các bạn chưa biết rằng mình nên bắt đầu từ đâu hoặc các bạn là những người đã tìm hiểu về ai về machining một thời gian rồi nhưng các bạn vẫn chưa thực sự hệ thống hóa được toàn bộ kiến thức của mình thì video này của mình là dành cho các bạn trong video này mình xin tổng hợp và tóm tắt toàn bộ các thật toán machin ling cơ bản và phổ biến nhất mà bất kỳ ai khi muốn học về ai học về machining đều cần phải nắm rõ do video này mình hướng đến đối tượng là cả những bạn chưa từng tiếp xúc với ai bao giờ do đó thì mình sẽ cố gắng giải thích tất cả mọi thứ ở mức đơn giản nhất có thể những cái thứ phức tạp như là công thức Toán học thì mình sẽ không đề cập đến 

In [53]:
preview_chunks(rag_chunks, start_index=5, n=3)

ULE78ME1ckQ_chunk_0006 | 346.32s - 446.04s
447 words | 99.72s
Units: ['unit_0016', 'unit_0017', 'unit_0018', 'unit_0019']
------------------------------------------------------------------------------------------------------------------------
ứng với y = 0 và Senior tương ứng với y = 1 thì chúng ta có thể trực văn hóa dữ liệu của chúng ta lên không gian hai chiều như các bạn có thể nhìn thấy trên màn hình như các bạn có thể nhìn thấy trong 11 nhân viên của công ty thì có năm người tương ứng với trình độ là Junior và sáu người tương ới với trình độ là CIO Giả sử chúng ta vẫn cố gắng muốn áp dụng thuật toán linear reaction vào trong bài toán này thì chúng ta sẽ được kết quả là đường thẳng màu đỏ như các bạn có thể nhìn thấy trên màn hình đường thẳng này gặp một vài những vấn đề như sau Thứ nhất là nó không có đi qua gần những cái điểm rượu của chúng ta hay nói khác là gì hay nói cá khác là khoảng cách từ các điểm dữ liệu đến đường thẳng này là rất lớn vấn đề thứ hai là đường thẳng này nó

In [54]:
# Format timestamp để hiển thị sau này

# Hàm này dùng cho chatbot RAG sau này, ví dụ trả lời:

# Nguồn: 03:12 - 04:25
def format_timestamp(seconds: float) -> str:
    """
    Chuyển giây sang định dạng MM:SS hoặc HH:MM:SS.
    """
    seconds = int(seconds)

    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60

    if h > 0:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"

In [55]:
for chunk in rag_chunks[:12]:
    print(
        chunk["chunk_id"],
        format_timestamp(chunk["start"]),
        "-",
        format_timestamp(chunk["end"])
    )

ULE78ME1ckQ_chunk_0001 00:00 - 01:32
ULE78ME1ckQ_chunk_0002 01:12 - 02:35
ULE78ME1ckQ_chunk_0003 02:11 - 03:47
ULE78ME1ckQ_chunk_0004 03:21 - 05:01
ULE78ME1ckQ_chunk_0005 04:33 - 06:12
ULE78ME1ckQ_chunk_0006 05:46 - 07:26
ULE78ME1ckQ_chunk_0007 06:59 - 08:35
ULE78ME1ckQ_chunk_0008 08:09 - 09:43
ULE78ME1ckQ_chunk_0009 09:18 - 10:53
ULE78ME1ckQ_chunk_0010 10:28 - 12:03
ULE78ME1ckQ_chunk_0011 11:37 - 13:15
ULE78ME1ckQ_chunk_0012 12:49 - 14:41


Sau khi transcript được tái cấu trúc thành các semantic units, hệ thống tiếp tục thực hiện bước Hybrid Chunking để tạo ra các đoạn văn bản phù hợp cho RAG. Mỗi chunk được xây dựng bằng cách kết hợp nhiều semantic units liên tiếp dựa trên số lượng từ, thời lượng và thứ tự timestamp. Hệ thống sử dụng cơ chế overlap giữa các chunk nhằm hạn chế mất ngữ cảnh tại ranh giới phân đoạn.

Mỗi chunk bao gồm mã chunk, mã video, timestamp bắt đầu, timestamp kết thúc, nội dung văn bản, số lượng từ, thời lượng và danh sách các semantic units cấu thành. Các chunk này sẽ được sử dụng làm đơn vị đầu vào cho mô hình embedding và được lưu vào vector database để phục vụ truy xuất ngữ nghĩa khi người dùng đặt câu hỏi.

Giai đoạn 6: Embedding

Mục tiêu giai đoạn này:

Input:
rag_chunks từ Giai đoạn 5

Xử lý:
- Load 2 embedding models
- Embedding từng chunk
- Embedding câu hỏi người dùng
- Tính cosine similarity
- Trả về Top-K chunks liên quan nhất

Output:
- Embedding vectors
- Kết quả search thử với từng model
- So sánh model A và model B

Hai model dùng để so sánh:

Model A: intfloat/multilingual-e5-base
Model B: BAAI/bge-m3

Ở giai đoạn này chưa dùng Vector Database. Mình sẽ cho bạn test search trực tiếp bằng numpy trước để kiểm tra embedding có hoạt động đúng không.

In [56]:
!pip install -q sentence-transformers

In [57]:
import os
import json
import time
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

In [58]:
print("Số chunks:", len(rag_chunks))

for chunk in rag_chunks[:2]:
    print("=" * 100)
    print(chunk["chunk_id"])
    print(chunk["start"], "-", chunk["end"])
    print(chunk["num_words"], "words")
    print(chunk["text"][:500])

Số chunks: 22
ULE78ME1ckQ_chunk_0001
0.08 - 92.76
489 words
Xin chào các bạn Nếu các bạn là những người đang muốn bắt đầu tìm hiểu về ai hay là machining nhưng các bạn chưa biết rằng mình nên bắt đầu từ đâu hoặc các bạn là những người đã tìm hiểu về ai về machining một thời gian rồi nhưng các bạn vẫn chưa thực sự hệ thống hóa được toàn bộ kiến thức của mình thì video này của mình là dành cho các bạn trong video này mình xin tổng hợp và tóm tắt toàn bộ các thật toán machin ling cơ bản và phổ biến nhất mà bất kỳ ai khi muốn học về ai học về machining đều c
ULE78ME1ckQ_chunk_0002
72.6 - 155.32
491 words
tiên là các tật Toán học có giám sát và các tật toán học không có giám sát là những tật toán được sử dụng rộng rãi và phổ biến nhất sự khác biệt chủ yếu của hai nhóm này đến từ việc là các Tộ Toán học có giám sát thì đều được huấn luyện với dữ liệu có đánh nhãn theo còn các Tộ toán học không giám sát thì đều được huấn luyện với dữ liệu không được đánh nhãn Thế nào là dữ liệu được đánh nhãn

In [59]:
# Hàm format text cho embedding
def prepare_texts_for_embedding(chunks: list, model_type: str = "e5") -> list:
    """
    Chuẩn bị text chunks trước khi embedding.

    model_type:
    - "e5": thêm prefix passage:
    - "bge": giữ nguyên hoặc thêm format nhẹ
    """
    texts = []

    for chunk in chunks:
        text = chunk["text"].strip()

        if model_type == "e5":
            text = "passage: " + text
        elif model_type == "bge":
            text = text
        else:
            text = text

        texts.append(text)

    return texts

In [60]:
# Hàm load embedding model
def load_embedding_model(model_name: str):
    """
    Load embedding model từ Hugging Face bằng sentence-transformers.
    """
    print(f"Đang load model: {model_name}")
    start_time = time.time()

    model = SentenceTransformer(model_name)

    elapsed = time.time() - start_time
    print(f"Load xong model sau {elapsed:.2f}s")

    return model

In [61]:
# Hàm tạo embedding cho chunks
def embed_chunks(
    model,
    chunks: list,
    model_type: str = "e5",
    batch_size: int = 8
) -> dict:
    """
    Embedding toàn bộ RAG chunks.

    Output:
    {
        "embeddings": np.ndarray,
        "embedding_time": ...,
        "chunk_ids": [...]
    }
    """
    texts = prepare_texts_for_embedding(chunks, model_type=model_type)
    chunk_ids = [chunk["chunk_id"] for chunk in chunks]

    start_time = time.time()

    embeddings = model.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    elapsed = time.time() - start_time

    return {
        "embeddings": embeddings,
        "embedding_time": elapsed,
        "chunk_ids": chunk_ids,
        "num_chunks": len(chunks),
        "embedding_dim": embeddings.shape[1]
    }

In [62]:
# Hàm embedding câu hỏi
def embed_query(model, query: str, model_type: str = "e5") -> np.ndarray:
    """
    Embedding câu hỏi người dùng.

    Với E5:
    query cần prefix "query: "

    Với BGE:
    có thể dùng trực tiếp.
    """
    query = query.strip()

    if model_type == "e5":
        query_text = "query: " + query
    elif model_type == "bge":
        query_text = query
    else:
        query_text = query

    query_embedding = model.encode(
        [query_text],
        convert_to_numpy=True,
        normalize_embeddings=True
    )[0]

    return query_embedding

In [63]:
# Load và embedding bằng Model A: multilingual-e5-base
E5_MODEL_NAME = "intfloat/multilingual-e5-base"

e5_model = load_embedding_model(E5_MODEL_NAME)

e5_result = embed_chunks(
    model=e5_model,
    chunks=rag_chunks,
    model_type="e5",
    batch_size=8
)

print("E5 embedding result:")
print("Num chunks:", e5_result["num_chunks"])
print("Embedding dim:", e5_result["embedding_dim"])
print("Embedding time:", round(e5_result["embedding_time"], 2), "s")
print("Embedding shape:", e5_result["embeddings"].shape)

Đang load model: intfloat/multilingual-e5-base


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Load xong model sau 10.76s


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

E5 embedding result:
Num chunks: 22
Embedding dim: 768
Embedding time: 1.6 s
Embedding shape: (22, 768)


In [64]:
# Load và embedding bằng Model B: BAAI/bge-m3

# Model này nặng hơn E5, nên nếu Kaggle chạy lâu là bình thường.

BGE_MODEL_NAME = "BAAI/bge-m3"

bge_model = load_embedding_model(BGE_MODEL_NAME)

bge_result = embed_chunks(
    model=bge_model,
    chunks=rag_chunks,
    model_type="bge",
    batch_size=4
)

print("BGE-M3 embedding result:")
print("Num chunks:", bge_result["num_chunks"])
print("Embedding dim:", bge_result["embedding_dim"])
print("Embedding time:", round(bge_result["embedding_time"], 2), "s")
print("Embedding shape:", bge_result["embeddings"].shape)

Đang load model: BAAI/bge-m3


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Load xong model sau 20.67s


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

BGE-M3 embedding result:
Num chunks: 22
Embedding dim: 1024
Embedding time: 2.08 s
Embedding shape: (22, 1024)


In [65]:
# Tạo QA chunk để tách chunk quá dài

qa_chunks = create_hybrid_chunks(
    semantic_units=semantic_units,
    video_id=video_id,
    target_words=220,
    min_words=120,
    max_words=280,
    max_duration=90.0,
    overlap_units=1
)

print("Số summary/rag chunks:", len(rag_chunks))
print("Số QA chunks:", len(qa_chunks))

for chunk in qa_chunks[:5]:
    print("=" * 100)
    print(f"{chunk['chunk_id']} | {chunk['start']}s - {chunk['end']}s")
    print(f"{chunk['num_words']} words | {chunk['duration']}s")
    print("Units:", chunk["unit_ids"])
    print(chunk["text"][:700])

Số summary/rag chunks: 22
Số QA chunks: 59
ULE78ME1ckQ_chunk_0001 | 0.08s - 49.52s
249 words | 49.44s
Units: ['unit_0001', 'unit_0002']
Xin chào các bạn Nếu các bạn là những người đang muốn bắt đầu tìm hiểu về ai hay là machining nhưng các bạn chưa biết rằng mình nên bắt đầu từ đâu hoặc các bạn là những người đã tìm hiểu về ai về machining một thời gian rồi nhưng các bạn vẫn chưa thực sự hệ thống hóa được toàn bộ kiến thức của mình thì video này của mình là dành cho các bạn trong video này mình xin tổng hợp và tóm tắt toàn bộ các thật toán machin ling cơ bản và phổ biến nhất mà bất kỳ ai khi muốn học về ai học về machining đều cần phải nắm rõ do video này mình hướng đến đối tượng là cả những bạn chưa từng tiếp xúc với ai bao giờ do đó thì mình sẽ cố gắng giải thích tất cả mọi thứ ở mức đơn giản nhất có thể những cái thứ phức
ULE78ME1ckQ_chunk_0002 | 23.6s - 74.2s
243 words | 50.6s
Units: ['unit_0002', 'unit_0003']
cả những bạn chưa từng tiếp xúc với ai bao giờ do đó thì mình sẽ cố gắng

In [66]:
qa_chunks_output = {
    "video_metadata": metadata,
    "chunking": {
        "type": "qa_chunks",
        "target_words": 220,
        "min_words": 120,
        "max_words": 280,
        "max_duration": 90.0,
        "overlap_units": 1
    },
    "num_qa_chunks": len(qa_chunks),
    "chunks": qa_chunks
}

qa_chunks_path = DATA_DIR / f"{video_id}_qa_chunks.json"

with open(qa_chunks_path, "w", encoding="utf-8") as f:
    json.dump(qa_chunks_output, f, ensure_ascii=False, indent=2)

print("Đã lưu QA chunks:", qa_chunks_path)

Đã lưu QA chunks: /kaggle/working/video_rag_project/data/ULE78ME1ckQ_qa_chunks.json


In [67]:
from pathlib import Path
import numpy as np

EMBEDDING_DIR = BASE_DIR / "embeddings"
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

print("EMBEDDING_DIR:", EMBEDDING_DIR)

EMBEDDING_DIR: /kaggle/working/video_rag_project/embeddings


In [68]:
# embedding qa chunk

qa_bge_result = embed_chunks(
    model=bge_model,
    chunks=qa_chunks,
    model_type="bge",
    batch_size=4
)

qa_bge_embeddings = qa_bge_result["embeddings"]

print("QA BGE embedding result:")
print("Num QA chunks:", qa_bge_result["num_chunks"])
print("Embedding dim:", qa_bge_result["embedding_dim"])
print("Embedding time:", round(qa_bge_result["embedding_time"], 2), "s")
print("Embedding shape:", qa_bge_embeddings.shape)

Batches:   0%|          | 0/15 [00:00<?, ?it/s]

QA BGE embedding result:
Num QA chunks: 59
Embedding dim: 1024
Embedding time: 2.81 s
Embedding shape: (59, 1024)


In [69]:
qa_bge_embedding_path = EMBEDDING_DIR / f"{video_id}_qa_bge_m3_embeddings.npy"
np.save(qa_bge_embedding_path, qa_bge_embeddings)

print("Đã lưu QA BGE embeddings:", qa_bge_embedding_path)

Đã lưu QA BGE embeddings: /kaggle/working/video_rag_project/embeddings/ULE78ME1ckQ_qa_bge_m3_embeddings.npy


In [70]:
!pip install -q rank-bm25

In [71]:
from rank_bm25 import BM25Okapi
import re
import numpy as np

def simple_tokenize(text: str):
    """
    Tokenize đơn giản cho tiếng Việt + tiếng Anh.
    Dùng cho BM25 keyword search.
    """
    text = text.lower()
    text = re.sub(r"[^\w\sÀ-ỹ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()


def build_bm25_index(chunks):
    """
    Tạo BM25 index từ danh sách chunks.
    """
    corpus_tokens = [simple_tokenize(chunk["text"]) for chunk in chunks]
    bm25 = BM25Okapi(corpus_tokens)
    return bm25


def bm25_search(query, bm25, chunks, top_k=10):
    """
    Search bằng BM25.
    """
    query_tokens = simple_tokenize(query)
    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "chunk_index": int(idx),
            "rank": rank,
            "bm25_score": float(scores[idx]),
            "chunk": chunks[idx]
        })

    return results

In [72]:
# Build BM25 index
qa_bm25 = build_bm25_index(qa_chunks)

print("Đã tạo BM25 index cho QA chunks:", len(qa_chunks))

Đã tạo BM25 index cho QA chunks: 59


In [73]:
# Dense search trả về index

# Bạn đang có hàm cũ:

# search_top_k_chunks(...)

# Hàm đó trả về kết quả đầy đủ, nhưng để gộp với BM25 thì cần trả về chunk_index.

# Vị trí đặt code

# Đặt sau BM25 functions.

# Cell mới: Dense search indices
def dense_search_indices(
    query,
    model,
    embeddings,
    chunks,
    model_type="bge",
    top_k=10
):
    """
    Dense search bằng embedding.
    Trả về index để gộp với BM25 bằng RRF.
    """
    query_embedding = embed_query(
        model=model,
        query=query,
        model_type=model_type
    )

    scores = np.dot(embeddings, query_embedding)
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        results.append({
            "chunk_index": int(idx),
            "rank": rank,
            "dense_score": float(scores[idx]),
            "chunk": chunks[idx]
        })

    return results

In [74]:
# RRF Fusion để gộp Dense + BM25
# Vị trí đặt code

# Đặt ngay sau dense_search_indices.

def rrf_fusion(
    dense_results,
    bm25_results,
    chunks,
    k=60,
    top_k=5
):
    """
    Gộp kết quả Dense Search và BM25 bằng Reciprocal Rank Fusion.
    Không cộng trực tiếp score vì dense_score và bm25_score khác thang đo.
    """
    fused_scores = {}

    # Dense results
    for item in dense_results:
        idx = item["chunk_index"]
        rank = item["rank"]
        fused_scores[idx] = fused_scores.get(idx, 0) + 1 / (k + rank)

    # BM25 results
    for item in bm25_results:
        idx = item["chunk_index"]
        rank = item["rank"]
        fused_scores[idx] = fused_scores.get(idx, 0) + 1 / (k + rank)

    sorted_items = sorted(
        fused_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    results = []

    for rank, (idx, score) in enumerate(sorted_items, start=1):
        chunk = chunks[idx]

        results.append({
            "rank": rank,
            "rrf_score": float(score),
            "chunk_index": int(idx),
            "chunk_id": chunk["chunk_id"],
            "start": chunk["start"],
            "end": chunk["end"],
            "text": chunk["text"],
            "num_words": chunk["num_words"],
            "duration": chunk["duration"],
            "unit_ids": chunk["unit_ids"]
        })

    return results

In [75]:
# Hàm Hybrid Search chính
# Vị trí đặt code

# Đặt ngay sau rrf_fusion.

def hybrid_search(
    query,
    chunks,
    dense_model,
    dense_embeddings,
    bm25,
    model_type="bge",
    dense_top_k=10,
    bm25_top_k=10,
    final_top_k=5
):
    """
    Hybrid Search:
    1. Dense search bằng embedding
    2. BM25 keyword search
    3. Gộp bằng RRF
    """

    dense_results = dense_search_indices(
        query=query,
        model=dense_model,
        embeddings=dense_embeddings,
        chunks=chunks,
        model_type=model_type,
        top_k=dense_top_k
    )

    bm25_results = bm25_search(
        query=query,
        bm25=bm25,
        chunks=chunks,
        top_k=bm25_top_k
    )

    final_results = rrf_fusion(
        dense_results=dense_results,
        bm25_results=bm25_results,
        chunks=chunks,
        k=60,
        top_k=final_top_k
    )

    return final_results

In [76]:
def is_retrieval_confident(results, min_rrf_score=0.03):
    """
    Kiểm tra kết quả retrieval có đủ tin cậy không.

    Nếu RRF score của top-1 quá thấp, hệ thống xem như
    không tìm thấy nội dung đủ chắc chắn trong video.
    """
    if not results:
        return False

    top_score = results[0].get("rrf_score", 0)

    return top_score >= min_rrf_score

In [77]:
# Hàm cosine similarity search

# Vì embeddings đã được normalize, cosine similarity có thể tính bằng dot product.

def search_top_k_chunks(
    query: str,
    model,
    chunk_embeddings: np.ndarray,
    chunks: list,
    model_type: str = "e5",
    top_k: int = 3
) -> list:
    """
    Search Top-K chunks liên quan nhất với câu hỏi.
    """
    query_embedding = embed_query(
        model=model,
        query=query,
        model_type=model_type
    )

    scores = np.dot(chunk_embeddings, query_embedding)

    top_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for rank, idx in enumerate(top_indices, start=1):
        chunk = chunks[idx]

        results.append({
            "rank": rank,
            "score": float(scores[idx]),
            "chunk_id": chunk["chunk_id"],
            "start": chunk["start"],
            "end": chunk["end"],
            "text": chunk["text"],
            "num_words": chunk["num_words"],
            "duration": chunk["duration"],
            "unit_ids": chunk["unit_ids"]
        })

    return results

In [78]:
# Hàm hiển thị kết quả search

def format_timestamp(seconds: float) -> str:
    seconds = int(seconds)
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60

    if h > 0:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"

In [79]:
def print_search_results(results: list, max_chars: int = 900):
    """
    In kết quả search dễ đọc.
    """
    for item in results:
        print("=" * 120)
        print(f"Rank {item['rank']} | Score: {item['score']:.4f}")
        print(f"Chunk: {item['chunk_id']}")
        print(f"Time: {format_timestamp(item['start'])} - {format_timestamp(item['end'])}")
        print(f"Words: {item['num_words']} | Duration: {item['duration']}s")
        print(f"Units: {item['unit_ids']}")
        print("-" * 120)
        print(item["text"][:max_chars])
        if len(item["text"]) > max_chars:
            print("...")

In [80]:
def print_hybrid_results(results, max_chars=900):
    """
    In kết quả Hybrid Search: Dense + BM25 + RRF.
    """
    for item in results:
        print("=" * 120)
        print(f"Rank {item['rank']} | RRF Score: {item['rrf_score']:.4f}")
        print(f"Chunk: {item['chunk_id']}")
        print(f"Time: {format_timestamp(item['start'])} - {format_timestamp(item['end'])}")
        print(f"Words: {item['num_words']} | Duration: {item['duration']}s")
        print(f"Units: {item['unit_ids']}")
        print("-" * 120)
        print(item["text"][:max_chars])

        if len(item["text"]) > max_chars:
            print("...")

## Test đúng luồng mới: QA chunks + BGE-M3 + BM25 + RRF

Không dùng lại các cell so sánh E5/BGE trên `rag_chunks` dài cho Chat QA.

In [81]:
# Kiểm tra nhanh biến của luồng mới
print("Số rag_chunks:", len(rag_chunks))
print("Số qa_chunks:", len(qa_chunks))
print("QA embedding shape:", qa_bge_embeddings.shape)
print("BM25 index ready:", qa_bm25 is not None)

print("\nVí dụ QA chunk đầu:")
print("chunk_id:", qa_chunks[0]["chunk_id"])
print("num_words:", qa_chunks[0]["num_words"])
print("time:", format_timestamp(qa_chunks[0]["start"]), "-", format_timestamp(qa_chunks[0]["end"]))
print(qa_chunks[0]["text"][:500])


Số rag_chunks: 22
Số qa_chunks: 59
QA embedding shape: (59, 1024)
BM25 index ready: True

Ví dụ QA chunk đầu:
chunk_id: ULE78ME1ckQ_chunk_0001
num_words: 249
time: 00:00 - 00:49
Xin chào các bạn Nếu các bạn là những người đang muốn bắt đầu tìm hiểu về ai hay là machining nhưng các bạn chưa biết rằng mình nên bắt đầu từ đâu hoặc các bạn là những người đã tìm hiểu về ai về machining một thời gian rồi nhưng các bạn vẫn chưa thực sự hệ thống hóa được toàn bộ kiến thức của mình thì video này của mình là dành cho các bạn trong video này mình xin tổng hợp và tóm tắt toàn bộ các thật toán machin ling cơ bản và phổ biến nhất mà bất kỳ ai khi muốn học về ai học về machining đều c


In [82]:
test_queries = [
    "AI là gì?",
    "Classification là gì?",
    "Overfitting là gì?"
]

all_hybrid_results = []

for q in test_queries:
    print("\n" + "#" * 120)
    print("QUERY:", q)
    print("#" * 120)

    hybrid_results = hybrid_search(
        query=q,
        chunks=qa_chunks,
        dense_model=bge_model,
        dense_embeddings=qa_bge_embeddings,
        bm25=qa_bm25,
        model_type="bge",
        dense_top_k=10,
        bm25_top_k=10,
        final_top_k=3
    )

    confident = is_retrieval_confident(
        hybrid_results,
        min_rrf_score=0.03
    )

    if not confident:
        print("Không tìm thấy nội dung đủ chắc chắn trong video.")
        print("Top retrieved chunks để debug:")
        print_hybrid_results(hybrid_results, max_chars=500)
    else:
        print_hybrid_results(hybrid_results, max_chars=700)

    all_hybrid_results.append({
        "query": q,
        "confident": confident,
        "results": hybrid_results
    })


########################################################################################################################
QUERY: AI là gì?
########################################################################################################################
Rank 1 | RRF Score: 0.0328
Chunk: ULE78ME1ckQ_chunk_0001
Time: 00:00 - 00:49
Words: 249 | Duration: 49.44s
Units: ['unit_0001', 'unit_0002']
------------------------------------------------------------------------------------------------------------------------
Xin chào các bạn Nếu các bạn là những người đang muốn bắt đầu tìm hiểu về ai hay là machining nhưng các bạn chưa biết rằng mình nên bắt đầu từ đâu hoặc các bạn là những người đã tìm hiểu về ai về machining một thời gian rồi nhưng các bạn vẫn chưa thực sự hệ thống hóa được toàn bộ kiến thức của mình thì video này của mình là dành cho các bạn trong video này mình xin tổng hợp và tóm tắt toàn bộ các thật toán machin ling cơ bản và phổ biến nhất mà bất kỳ ai khi muốn học về ai 

In [83]:
# Tạo bảng so sánh Top-1 của Hybrid Search
import pandas as pd

def build_hybrid_retrieval_table(queries, chunks, dense_model, dense_embeddings, bm25):
    rows = []

    for query in queries:
        results = hybrid_search(
            query=query,
            chunks=chunks,
            dense_model=dense_model,
            dense_embeddings=dense_embeddings,
            bm25=bm25,
            model_type="bge",
            dense_top_k=10,
            bm25_top_k=10,
            final_top_k=3
        )

        top1 = results[0]

        rows.append({
            "query": query,
            "top1_chunk": top1["chunk_id"],
            "rrf_score": round(top1["rrf_score"], 4),
            "time": f"{format_timestamp(top1['start'])} - {format_timestamp(top1['end'])}",
            "num_words": top1["num_words"],
            "preview": top1["text"][:180]
        })

    return pd.DataFrame(rows)

hybrid_comparison_df = build_hybrid_retrieval_table(
    queries=test_queries,
    chunks=qa_chunks,
    dense_model=bge_model,
    dense_embeddings=qa_bge_embeddings,
    bm25=qa_bm25
)

hybrid_comparison_df


,query,top1_chunk,rrf_score,time,num_words,preview
0,AI là gì?,ULE78ME1ckQ_chunk_0001,0.0328,00:00 - 00:49,249,Xin chào các bạn Nếu các bạn là những người đa...
1,Classification là gì?,ULE78ME1ckQ_chunk_0048,0.0328,20:51 - 21:39,240,với bài toán classification bài toàn phân loại...
2,Overfitting là gì?,ULE78ME1ckQ_chunk_0019,0.0294,08:09 - 08:58,217,"nhỏ hơn 0,5 thì người ta sẽ coi như output là ..."


In [84]:
# Lưu kết quả hybrid retrieval để đưa vào báo cáo
HYBRID_RESULT_DIR = BASE_DIR / "hybrid_results"
HYBRID_RESULT_DIR.mkdir(parents=True, exist_ok=True)

hybrid_results_path = HYBRID_RESULT_DIR / f"{video_id}_hybrid_search_results.json"
hybrid_table_path = HYBRID_RESULT_DIR / f"{video_id}_hybrid_search_table.csv"

with open(hybrid_results_path, "w", encoding="utf-8") as f:
    json.dump(all_hybrid_results, f, ensure_ascii=False, indent=2)

hybrid_comparison_df.to_csv(hybrid_table_path, index=False, encoding="utf-8-sig")

print("Đã lưu hybrid results:", hybrid_results_path)
print("Đã lưu hybrid table:", hybrid_table_path)


Đã lưu hybrid results: /kaggle/working/video_rag_project/hybrid_results/ULE78ME1ckQ_hybrid_search_results.json
Đã lưu hybrid table: /kaggle/working/video_rag_project/hybrid_results/ULE78ME1ckQ_hybrid_search_table.csv


In [85]:
## Lưu embedding summary chunks nếu cần so sánh baseline

# Cell này chỉ để lưu kết quả E5/BGE trên `rag_chunks`, không dùng cho Chat QA cuối.

EMBEDDING_DIR = BASE_DIR / "embeddings"
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

video_id = metadata["video_id"]

e5_embedding_path = EMBEDDING_DIR / f"{video_id}_e5_embeddings.npy"
bge_embedding_path = EMBEDDING_DIR / f"{video_id}_bge_m3_embeddings.npy"

np.save(e5_embedding_path, e5_result["embeddings"])
np.save(bge_embedding_path, bge_result["embeddings"])

print("Đã lưu E5 embeddings:", e5_embedding_path)
print("Đã lưu BGE-M3 embeddings:", bge_embedding_path)


Đã lưu E5 embeddings: /kaggle/working/video_rag_project/embeddings/ULE78ME1ckQ_e5_embeddings.npy
Đã lưu BGE-M3 embeddings: /kaggle/working/video_rag_project/embeddings/ULE78ME1ckQ_bge_m3_embeddings.npy


In [86]:
embedding_metadata = {
    "video_id": video_id,
    "num_chunks": len(rag_chunks),
    "models": {
        "e5": {
            "model_name": E5_MODEL_NAME,
            "embedding_dim": int(e5_result["embedding_dim"]),
            "embedding_time": round(e5_result["embedding_time"], 4),
            "embedding_path": str(e5_embedding_path)
        },
        "bge_m3": {
            "model_name": BGE_MODEL_NAME,
            "embedding_dim": int(bge_result["embedding_dim"]),
            "embedding_time": round(bge_result["embedding_time"], 4),
            "embedding_path": str(bge_embedding_path)
        }
    },
    "chunk_ids": [chunk["chunk_id"] for chunk in rag_chunks]
}

embedding_metadata_path = EMBEDDING_DIR / f"{video_id}_embedding_metadata.json"

with open(embedding_metadata_path, "w", encoding="utf-8") as f:
    json.dump(embedding_metadata, f, ensure_ascii=False, indent=2)

print("Đã lưu embedding metadata:", embedding_metadata_path)

Đã lưu embedding metadata: /kaggle/working/video_rag_project/embeddings/ULE78ME1ckQ_embedding_metadata.json


In [87]:
query = "Overfitting là gì?"

results = hybrid_search(
    query=query,
    chunks=qa_chunks,
    dense_model=bge_model,
    dense_embeddings=qa_bge_embeddings,
    bm25=qa_bm25,
    model_type="bge",
    dense_top_k=10,
    bm25_top_k=10,
    final_top_k=3
)

if not is_retrieval_confident(results, min_rrf_score=0.03):
    print("Không tìm thấy nội dung đủ chắc chắn trong video.")
else:
    print_hybrid_results(results)

Không tìm thấy nội dung đủ chắc chắn trong video.


Ở giai đoạn embedding, hệ thống sử dụng hai mô hình embedding đa ngôn ngữ là intfloat/multilingual-e5-base và BAAI/bge-m3 để mã hóa các RAG chunks thành vector số. Mỗi chunk transcript sau khi được xử lý và chia đoạn sẽ được đưa qua mô hình embedding để tạo vector đại diện ngữ nghĩa. Khi người dùng đặt câu hỏi, câu hỏi cũng được mã hóa bằng cùng mô hình embedding, sau đó hệ thống tính độ tương đồng cosine giữa vector câu hỏi và vector của các chunk để tìm ra những đoạn nội dung liên quan nhất.

Việc sử dụng hai mô hình embedding giúp hệ thống có cơ sở so sánh chất lượng truy xuất giữa một mô hình baseline nhẹ hơn là multilingual-e5-base và một mô hình mạnh hơn là bge-m3. Kết quả truy xuất được đánh giá thông qua top-k chunks, điểm similarity, timestamp tương ứng và mức độ phù hợp với câu hỏi.

=========================TÓM TẮT==================================

SO SÁNH 2 MODEL LLM TÓM TẮT: Qwen/Qwen2.5-3B-Instruct | google/gemma-2-2b-it

vì nó là instruct model nhỏ, dễ chạy hơn model 7B trên Kaggle, phù hợp để so sánh với Qwen 3B. Qwen2.5 có các bản instruct từ 0.5B đến 72B, còn Qwen/Qwen2.5-3B-Instruct dùng được trực tiếp với Transformers; Gemma 2 2B IT cũng có hướng dẫn dùng với Transformers trên Hugging Face

In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import torch
import json
import time
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

In [103]:
# ================================
# Summarization Models
# ================================

QWEN_SUMMARY_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
GEMMA_SUMMARY_MODEL_NAME = "google/gemma-2-2b-it"

print("Model 1:", QWEN_SUMMARY_MODEL_NAME)
print("Model 2:", GEMMA_SUMMARY_MODEL_NAME)

Model 1: Qwen/Qwen2.5-3B-Instruct
Model 2: google/gemma-2-2b-it


In [102]:
SUMMARIZATION_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

sum_tokenizer, sum_model = load_summarization_llm(SUMMARIZATION_MODEL_NAME)

Đang load model: Qwen/Qwen2.5-3B-Instruct


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Load model xong.


In [104]:
def load_summarization_llm(model_name: str, use_4bit: bool = True):
    """
    Load LLM local để tóm tắt.
    Dùng 4-bit quantization để giảm VRAM trên Kaggle.
    Hỗ trợ Qwen và Gemma.
    """
    print("Đang load model:", model_name)

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=True
    )

    if use_4bit:
        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4"
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True
        )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )

    print("Load model xong:", model_name)
    return tokenizer, model

In [105]:
qwen_tokenizer, qwen_model = load_summarization_llm(
    QWEN_SUMMARY_MODEL_NAME,
    use_4bit=True
)

Đang load model: Qwen/Qwen2.5-3B-Instruct


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Load model xong: Qwen/Qwen2.5-3B-Instruct


In [106]:
def generate_llm_response(
    prompt: str,
    tokenizer,
    model,
    max_new_tokens: int = 512,
    temperature: float = 0.2
) -> str:
    """
    Sinh phản hồi từ LLM.
    """
    messages = [
        {
            "role": "system",
            "content": (
                "Bạn là trợ lý AI chuyên tóm tắt video bài giảng IT/AI. "
                "Chỉ tóm tắt dựa trên nội dung được cung cấp. "
                "Không bịa thêm thông tin không có trong transcript. "
                "Trả lời bằng tiếng Việt, giữ nguyên thuật ngữ kỹ thuật tiếng Anh khi cần."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True if temperature > 0 else False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return response.strip()

In [108]:
# Tóm tắt từng chunk
def summarize_single_chunk(chunk, tokenizer, model):
    """
    Tóm tắt một RAG chunk.
    """
    start_time = format_timestamp(chunk["start"])
    end_time = format_timestamp(chunk["end"])

    prompt = f"""
Bạn hãy tóm tắt đoạn transcript video sau.

Yêu cầu:
- Tóm tắt bằng tiếng Việt.
- Giữ nguyên thuật ngữ kỹ thuật tiếng Anh nếu cần.
- Không bịa thêm nội dung ngoài transcript.
- Nêu rõ ý chính của đoạn.
- Nếu đoạn có nhiều thuật toán/khái niệm, hãy liệt kê ngắn gọn.
- Độ dài: 3-5 câu.

Thông tin đoạn:
- Chunk ID: {chunk["chunk_id"]}
- Thời gian: {start_time} - {end_time}

Transcript:
{chunk["text"]}
"""

    summary = generate_llm_response(
        prompt=prompt,
        tokenizer=tokenizer,
        model=model,
        max_new_tokens=350,
        temperature=0.2
    )

    return {
        "chunk_id": chunk["chunk_id"],
        "video_id": chunk["video_id"],
        "start": chunk["start"],
        "end": chunk["end"],
        "start_time": start_time,
        "end_time": end_time,
        "num_words": chunk["num_words"],
        "summary": summary,
        "source_text": chunk["text"]
    }

In [ ]:
def create_chapters_from_chunk_summaries(chunk_summaries, chunks_per_chapter=3):
    """
    Gom nhiều chunk summary thành chapter.
    Mỗi chapter gồm khoảng 3 chunks liên tiếp.
    """
    chapters = []

    for i in range(0, len(chunk_summaries), chunks_per_chapter):
        group = chunk_summaries[i:i + chunks_per_chapter]

        if not group:
            continue

        chapter_id = f"chapter_{len(chapters) + 1:03d}"

        chapter = {
            "chapter_id": chapter_id,
            "start": group[0]["start"],
            "end": group[-1]["end"],
            "start_time": group[0]["start_time"],
            "end_time": group[-1]["end_time"],
            "chunk_ids": [item["chunk_id"] for item in group],
            "chunk_summaries": group
        }

        chapters.append(chapter)

    return chapters

In [110]:
def summarize_single_chapter(chapter, tokenizer, model):
    """
    Tóm tắt một chapter dựa trên các chunk summaries.
    """
    combined_chunk_summaries = "\n\n".join([
        f"- {item['start_time']} - {item['end_time']}: {item['summary']}"
        for item in chapter["chunk_summaries"]
    ])

    prompt = f"""
Bạn hãy tóm tắt chương video sau dựa trên các tóm tắt chunk.

Yêu cầu:
- Đặt tiêu đề ngắn cho chương.
- Tóm tắt nội dung chương bằng tiếng Việt.
- Giữ nguyên thuật ngữ kỹ thuật tiếng Anh nếu cần.
- Không bịa thêm nội dung ngoài dữ liệu.
- Độ dài phần summary: 4-6 câu.
- Trả về đúng JSON với format:

{{
  "title": "...",
  "summary": "...",
  "key_points": ["...", "...", "..."]
}}

Thông tin chapter:
- Chapter ID: {chapter["chapter_id"]}
- Thời gian: {chapter["start_time"]} - {chapter["end_time"]}

Các chunk summaries:
{combined_chunk_summaries}
"""

    response = generate_llm_response(
        prompt=prompt,
        tokenizer=tokenizer,
        model=model,
        max_new_tokens=600,
        temperature=0.2
    )

    return response

In [112]:
def summarize_final_video(metadata, chapter_summaries, tokenizer, model):
    """
    Tạo tóm tắt cuối cùng cho toàn bộ video.
    """
    chapter_text = "\n\n".join([
        f"- {item['start_time']} - {item['end_time']}:\n{item['raw_response']}"
        for item in chapter_summaries
    ])

    prompt = f"""
Bạn hãy tạo bản tóm tắt cuối cùng cho toàn bộ video.

Yêu cầu:
- Tóm tắt bằng tiếng Việt.
- Giữ nguyên thuật ngữ kỹ thuật tiếng Anh nếu cần.
- Không bịa thêm thông tin ngoài nội dung được cung cấp.
- Tạo:
  1. Tóm tắt ngắn 5-6 câu.
  2. Tóm tắt chi tiết hơn.
  3. Các ý chính.
  4. Timeline nội dung theo chapter.
  5. Kết luận.
- Nếu có phần quảng cáo/giới thiệu khóa học ở cuối, hãy ghi rõ đó là phần giới thiệu/ngoài nội dung chính.

Thông tin video:
- Title: {metadata.get("title")}
- Channel: {metadata.get("channel")}
- Duration: {metadata.get("duration")} giây
- URL: {metadata.get("youtube_url")}

Chapter summaries:
{chapter_text}
"""

    final_summary = generate_llm_response(
        prompt=prompt,
        tokenizer=tokenizer,
        model=model,
        max_new_tokens=1200,
        temperature=0.2
    )

    return final_summary

In [114]:
def run_hierarchical_summarization(
    rag_chunks,
    metadata,
    tokenizer,
    model,
    model_name: str,
    chunks_per_chapter: int = 3
):
    """
    Chạy hierarchical summarization cho một model:
    1. Chunk summaries
    2. Chapter summaries
    3. Final summary
    """
    print("=" * 120)
    print("RUN SUMMARIZATION MODEL:", model_name)
    print("=" * 120)

    chunk_summaries = []

    for i, chunk in enumerate(rag_chunks, start=1):
        print(f"Đang tóm tắt chunk {i}/{len(rag_chunks)}: {chunk['chunk_id']}")

        chunk_summary = summarize_single_chunk(
            chunk=chunk,
            tokenizer=tokenizer,
            model=model
        )

        chunk_summaries.append(chunk_summary)

    chapters = create_chapters_from_chunk_summaries(
        chunk_summaries=chunk_summaries,
        chunks_per_chapter=chunks_per_chapter
    )

    chapter_summaries = []

    for i, chapter in enumerate(chapters, start=1):
        print(f"Đang tóm tắt chapter {i}/{len(chapters)}: {chapter['chapter_id']}")

        response = summarize_single_chapter(
            chapter=chapter,
            tokenizer=tokenizer,
            model=model
        )

        chapter_summary = {
            "chapter_id": chapter["chapter_id"],
            "start": chapter["start"],
            "end": chapter["end"],
            "start_time": chapter["start_time"],
            "end_time": chapter["end_time"],
            "chunk_ids": chapter["chunk_ids"],
            "raw_response": response
        }

        chapter_summaries.append(chapter_summary)

    final_summary = summarize_final_video(
        metadata=metadata,
        chapter_summaries=chapter_summaries,
        tokenizer=tokenizer,
        model=model
    )

    output = {
        "model_name": model_name,
        "video_metadata": metadata,
        "chunk_summaries": chunk_summaries,
        "chapter_summaries": chapter_summaries,
        "final_summary": final_summary
    }

    return output

In [ ]:
# Qwen


qwen_summary_output = run_hierarchical_summarization(
    rag_chunks=rag_chunks,
    metadata=metadata,
    tokenizer=qwen_tokenizer,
    model=qwen_model,
    model_name=QWEN_SUMMARY_MODEL_NAME,
    chunks_per_chapter=3
)

print(qwen_summary_output["final_summary"])

RUN SUMMARIZATION MODEL: Qwen/Qwen2.5-3B-Instruct
Đang tóm tắt chunk 1/22: ULE78ME1ckQ_chunk_0001
Đang tóm tắt chunk 2/22: ULE78ME1ckQ_chunk_0002
Đang tóm tắt chunk 3/22: ULE78ME1ckQ_chunk_0003
Đang tóm tắt chunk 4/22: ULE78ME1ckQ_chunk_0004
Đang tóm tắt chunk 5/22: ULE78ME1ckQ_chunk_0005
Đang tóm tắt chunk 6/22: ULE78ME1ckQ_chunk_0006
Đang tóm tắt chunk 7/22: ULE78ME1ckQ_chunk_0007
Đang tóm tắt chunk 8/22: ULE78ME1ckQ_chunk_0008
Đang tóm tắt chunk 9/22: ULE78ME1ckQ_chunk_0009
Đang tóm tắt chunk 10/22: ULE78ME1ckQ_chunk_0010
Đang tóm tắt chunk 11/22: ULE78ME1ckQ_chunk_0011
Đang tóm tắt chunk 12/22: ULE78ME1ckQ_chunk_0012
Đang tóm tắt chunk 13/22: ULE78ME1ckQ_chunk_0013
Đang tóm tắt chunk 14/22: ULE78ME1ckQ_chunk_0014
Đang tóm tắt chunk 15/22: ULE78ME1ckQ_chunk_0015
Đang tóm tắt chunk 16/22: ULE78ME1ckQ_chunk_0016
Đang tóm tắt chunk 17/22: ULE78ME1ckQ_chunk_0017
Đang tóm tắt chunk 18/22: ULE78ME1ckQ_chunk_0018
Đang tóm tắt chunk 19/22: ULE78ME1ckQ_chunk_0019
Đang tóm tắt chunk 20/22: UL

In [ ]:
SUMMARY_DIR = BASE_DIR / "summaries"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

qwen_summary_path = SUMMARY_DIR / f"{video_id}_qwen_summary.json"

with open(qwen_summary_path, "w", encoding="utf-8") as f:
    json.dump(qwen_summary_output, f, ensure_ascii=False, indent=2)

print("Đã lưu Qwen summary:", qwen_summary_path)

=======================================================================================================

In [89]:
# Chạy tóm tắt toàn bộ chunks
chunk_summaries = []

start_all = time.time()

for i, chunk in enumerate(rag_chunks, start=1):
    print("=" * 100)
    print(f"Đang tóm tắt chunk {i}/{len(rag_chunks)}: {chunk['chunk_id']}")
    print(f"Time: {format_timestamp(chunk['start'])} - {format_timestamp(chunk['end'])}")

    chunk_summary = summarize_single_chunk(
        chunk=chunk,
        tokenizer=sum_tokenizer,
        model=sum_model
    )

    chunk_summaries.append(chunk_summary)

    print(chunk_summary["summary"])

elapsed = time.time() - start_all
print("Hoàn thành chunk summaries sau:", round(elapsed, 2), "giây")

Đang tóm tắt chunk 1/22: ULE78ME1ckQ_chunk_0001
Time: 00:00 - 01:32
Dưới đây là tóm tắt nội dung transcript:

Máy học (Machine Learning) là một lĩnh vực của AI tập trung vào việc giúp máy tính tự học từ dữ liệu mà không cần lập trình cụ thể. Trong machining, có nhiều thuật toán khác nhau, nhưng có thể phân loại thành ba nhóm chính dựa trên phương pháp học:

1. Máy học có giám sát (Supervised Learning): Thuật toán này được huấn luyện với dữ liệu có đánh nhãn, tức là dữ liệu đã được phân loại và đánh dấu sẵn.

2. Máy học không giám sát (Unsupervised Learning): Thuật toán này không cần dữ liệu đánh nhãn, chỉ cần dữ liệu không được phân loại.

3. Máy học tăng cường (Reinforcement Learning): Thuật toán này học qua phản hồi từ môi trường, thông qua việc nhận biết và học từ hậu quả của hành động.

Trong machining, hai nhóm đầu tiên (máy học có giám sát và không giám sát) được sử dụng phổ biến nhất và có nhiều ứng dụng thực tế. Dữ liệu được đánh nhãn là dữ liệu đã được phân loại và đánh dấu sẵ

In [90]:
# Lưu chunk summaries
SUMMARY_DIR = BASE_DIR / "summaries"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)

chunk_summary_path = SUMMARY_DIR / f"{video_id}_chunk_summaries.json"

chunk_summary_output = {
    "video_metadata": metadata,
    "summary_type": "chunk_summary",
    "num_chunks": len(chunk_summaries),
    "chunk_summaries": chunk_summaries
}

with open(chunk_summary_path, "w", encoding="utf-8") as f:
    json.dump(chunk_summary_output, f, ensure_ascii=False, indent=2)

print("Đã lưu chunk summaries:", chunk_summary_path)

Đã lưu chunk summaries: /kaggle/working/video_rag_project/summaries/ULE78ME1ckQ_chunk_summaries.json


In [91]:
# Tạo chapter từ chunk summaries

# Tạm thời gom mỗi 3 chunks thành 1 chapter. Cách này ổn cho bản đầu.

def create_chapters_from_chunk_summaries(chunk_summaries, chunks_per_chapter=3):
    """
    Gom nhiều chunk summary thành chapter.
    Mỗi chapter gồm khoảng 3 chunks liên tiếp.
    """
    chapters = []

    for i in range(0, len(chunk_summaries), chunks_per_chapter):
        group = chunk_summaries[i:i + chunks_per_chapter]

        if not group:
            continue

        chapter_id = f"chapter_{len(chapters) + 1:03d}"

        chapter = {
            "chapter_id": chapter_id,
            "start": group[0]["start"],
            "end": group[-1]["end"],
            "start_time": group[0]["start_time"],
            "end_time": group[-1]["end_time"],
            "chunk_ids": [item["chunk_id"] for item in group],
            "chunk_summaries": group
        }

        chapters.append(chapter)

    return chapters

In [ ]:
chapters = create_chapters_from_chunk_summaries(
    chunk_summaries=chunk_summaries,
    chunks_per_chapter=3
)

print("Số chapters:", len(chapters))

for ch in chapters:
    print(ch["chapter_id"], ch["start_time"], "-", ch["end_time"], ch["chunk_ids"])

In [93]:
def summarize_single_chapter(chapter, tokenizer, model):
    """
    Tóm tắt một chapter dựa trên các chunk summaries.
    """
    combined_chunk_summaries = "\n\n".join([
        f"- {item['start_time']} - {item['end_time']}: {item['summary']}"
        for item in chapter["chunk_summaries"]
    ])

    prompt = f"""
Bạn hãy tóm tắt chương video sau dựa trên các tóm tắt chunk.

Yêu cầu:
- Đặt tiêu đề ngắn cho chương.
- Tóm tắt nội dung chương bằng tiếng Việt.
- Giữ nguyên thuật ngữ kỹ thuật tiếng Anh nếu cần.
- Không bịa thêm nội dung ngoài dữ liệu.
- Độ dài phần summary: 4-6 câu.
- Trả về đúng JSON với format:

{{
  "title": "...",
  "summary": "...",
  "key_points": ["...", "...", "..."]
}}

Thông tin chapter:
- Chapter ID: {chapter["chapter_id"]}
- Thời gian: {chapter["start_time"]} - {chapter["end_time"]}

Các chunk summaries:
{combined_chunk_summaries}
"""

    response = generate_llm_response(
        prompt=prompt,
        tokenizer=tokenizer,
        model=model,
        max_new_tokens=600,
        temperature=0.2
    )

    return response

In [94]:
chapter_summaries = []

for i, chapter in enumerate(chapters, start=1):
    print("=" * 100)
    print(f"Đang tóm tắt chapter {i}/{len(chapters)}: {chapter['chapter_id']}")
    print(f"Time: {chapter['start_time']} - {chapter['end_time']}")

    response = summarize_single_chapter(
        chapter=chapter,
        tokenizer=sum_tokenizer,
        model=sum_model
    )

    chapter_summary = {
        "chapter_id": chapter["chapter_id"],
        "start": chapter["start"],
        "end": chapter["end"],
        "start_time": chapter["start_time"],
        "end_time": chapter["end_time"],
        "chunk_ids": chapter["chunk_ids"],
        "raw_response": response
    }

    chapter_summaries.append(chapter_summary)

    print(response)

Đang tóm tắt chapter 1/8: chapter_001
Time: 00:00 - 03:47
{
  "title": "Giới thiệu về Máy Học và Thuật Toán Học Có Giám Sát",
  "summary": "Chương này giới thiệu về máy học và các thuật toán học có giám sát. Nó phân biệt giữa hai nhóm thuật toán học có và không có giám sát, với emphasis lớn hơn trên học có giám sát. Chương cũng đưa ra ví dụ về thuật toán hồi quy tuyến tính, một trong những thuật toán học có giám sát phổ biến nhất. Thông qua việc sử dụng dữ liệu đã đánh nhãn, chương giải thích cách máy học tự học từ dữ liệu mà không cần lập trình cụ thể.",
  "key_points": [
    "Máy học là một lĩnh vực của AI giúp máy tính tự học từ dữ liệu mà không cần lập trình cụ thể.",
    "Thuật toán học có giám sát và không giám sát là hai nhóm phổ biến trong học máy.",
    "Thuật toán học có giám sát phổ biến hơn và có nhiều ứng dụng thực tế hơn so với không giám sát."
  ]
}
Đang tóm tắt chapter 2/8: chapter_002
Time: 03:21 - 07:26
{
  "title": "Thuật toán Linear Regression và Logistic Regression

In [95]:
chapter_summary_path = SUMMARY_DIR / f"{video_id}_chapter_summaries.json"

chapter_summary_output = {
    "video_metadata": metadata,
    "summary_type": "chapter_summary",
    "num_chapters": len(chapter_summaries),
    "chapter_summaries": chapter_summaries
}

with open(chapter_summary_path, "w", encoding="utf-8") as f:
    json.dump(chapter_summary_output, f, ensure_ascii=False, indent=2)

print("Đã lưu chapter summaries:", chapter_summary_path)

Đã lưu chapter summaries: /kaggle/working/video_rag_project/summaries/ULE78ME1ckQ_chapter_summaries.json


In [96]:
def summarize_final_video(metadata, chapter_summaries, tokenizer, model):
    """
    Tạo tóm tắt cuối cùng cho toàn bộ video.
    """
    chapter_text = "\n\n".join([
        f"- {item['start_time']} - {item['end_time']}:\n{item['raw_response']}"
        for item in chapter_summaries
    ])

    prompt = f"""
Bạn hãy tạo bản tóm tắt cuối cùng cho toàn bộ video.

Yêu cầu:
- Tóm tắt bằng tiếng Việt.
- Giữ nguyên thuật ngữ kỹ thuật tiếng Anh nếu cần.
- Không bịa thêm thông tin ngoài nội dung được cung cấp.
- Tạo:
  1. Tóm tắt ngắn 5-6 câu.
  2. Tóm tắt chi tiết hơn.
  3. Các ý chính.
  4. Timeline nội dung theo chapter.
- Nếu có phần quảng cáo/giới thiệu khóa học ở cuối, hãy ghi rõ đó là phần giới thiệu/ngoài nội dung chính.

Thông tin video:
- Title: {metadata.get("title")}
- Channel: {metadata.get("channel")}
- Duration: {metadata.get("duration")} giây
- URL: {metadata.get("youtube_url")}

Chapter summaries:
{chapter_text}
"""

    final_summary = generate_llm_response(
        prompt=prompt,
        tokenizer=tokenizer,
        model=model,
        max_new_tokens=1200,
        temperature=0.2
    )

    return final_summary

In [97]:
final_video_summary = summarize_final_video(
    metadata=metadata,
    chapter_summaries=chapter_summaries,
    tokenizer=sum_tokenizer,
    model=sum_model
)

print(final_video_summary)

### Tóm tắt ngắn 5-6 câu

Bài giảng này giới thiệu các thuật toán học máy cơ bản như hồi quy tuyến tính, hồi quy logistic, SVM, KNN, Naive Bayes, phân loại email spam, ensemble learning, boosting, stacking, phân cụm, và nền tảng deep learning. Bài giảng cũng đề cập đến Docker, một công cụ quan trọng trong AI và công nghệ. Kết thúc video là lời mời tham gia khóa học về toán học.

### Tóm tắt chi tiết hơn

Bài giảng này bắt đầu bằng việc giới thiệu về máy học và các thuật toán học có giám sát. Sau đó, nó tập trung vào các thuật toán học có giám sát như hồi quy tuyến tính và hồi quy logistic. Bài giảng cũng thảo luận về SVM và KNN, hai thuật toán phân loại nhị phân. 

Tiếp theo, bài giảng giới thiệu về phân loại email spam và thuật toán Naive Bayes. Bài giảng cũng đề cập đến phương pháp ensemble learning, bao gồm boosting và stacking. 

Bài giảng sau đó tập trung vào quá trình huấn luyện mô hình boosting và stacking. Cuối cùng, bài giảng giới thiệu về phân cụm và Docker, một công cụ quan 

In [98]:
final_summary_path = SUMMARY_DIR / f"{video_id}_final_summary.json"

final_summary_output = {
    "video_metadata": metadata,
    "summary_type": "final_video_summary",
    "final_summary": final_video_summary,
    "chapter_summaries": chapter_summaries,
    "chunk_summary_path": str(chunk_summary_path),
    "chapter_summary_path": str(chapter_summary_path)
}

with open(final_summary_path, "w", encoding="utf-8") as f:
    json.dump(final_summary_output, f, ensure_ascii=False, indent=2)

print("Đã lưu final summary:", final_summary_path)

Đã lưu final summary: /kaggle/working/video_rag_project/summaries/ULE78ME1ckQ_final_summary.json


================================PREDICT SUMARY LLM=============================================

In [100]:
def display_video_summary(metadata, final_summary, chapter_summaries):
    print("=" * 120)
    print("VIDEO METADATA")
    print("=" * 120)
    print("Title:", metadata.get("title"))
    print("Channel:", metadata.get("channel"))
    print("URL:", metadata.get("youtube_url"))
    print("Duration:", metadata.get("duration"), "seconds")

    print("\n" + "=" * 120)
    print("FINAL SUMMARY")
    print("=" * 120)
    print(final_summary)

    print("\n" + "=" * 120)
    print("CHAPTER TIMELINE")
    print("=" * 120)

    for chapter in chapter_summaries:
        print(f"{chapter['start_time']} - {chapter['end_time']} | {chapter['chapter_id']}")
        print(chapter["raw_response"][:500])
        print("-" * 120)

In [101]:
display_video_summary(
    metadata=metadata,
    final_summary=final_video_summary,
    chapter_summaries=chapter_summaries
)

VIDEO METADATA
Title: Tất cả các thuật toán Machine Learning trong 23 phút
Channel: Việt Nguyễn AI
URL: https://www.youtube.com/watch?v=ULE78ME1ckQ
Duration: 1558 seconds

FINAL SUMMARY
### Tóm tắt ngắn 5-6 câu

Bài giảng này giới thiệu các thuật toán học máy cơ bản như hồi quy tuyến tính, hồi quy logistic, SVM, KNN, Naive Bayes, phân loại email spam, ensemble learning, boosting, stacking, phân cụm, và nền tảng deep learning. Bài giảng cũng đề cập đến Docker, một công cụ quan trọng trong AI và công nghệ. Kết thúc video là lời mời tham gia khóa học về toán học.

### Tóm tắt chi tiết hơn

Bài giảng này bắt đầu bằng việc giới thiệu về máy học và các thuật toán học có giám sát. Sau đó, nó tập trung vào các thuật toán học có giám sát như hồi quy tuyến tính và hồi quy logistic. Bài giảng cũng thảo luận về SVM và KNN, hai thuật toán phân loại nhị phân. 

Tiếp theo, bài giảng giới thiệu về phân loại email spam và thuật toán Naive Bayes. Bài giảng cũng đề cập đến phương pháp ensemble learning, 